# DSC@UIT 2026 — Task 1: LegalIR — **v3B: fine-tune reranker**
**Team: SGU_LegalMind** · Codabench: https://www.codabench.org/competitions/17715/

Pipeline truy vấn văn bản pháp luật tiếng Việt:

```
question ─┬─► BM25 (chunk)   ─┐
          └─► dense (chunk)  ─┴─► RRF fuse ─► top-N doc ─► cross-encoder rerank ─► top-5
```

## Layout Drive thực tế

```
[Shared with me] UIT DS Challenge/        ← owner: phatleh.224@gmail.com
└── Task 1/
    ├── LegalR - Public Test/             ← DỮ LIỆU (đường dẫn cố định trong Config)
    │     train.json · private-official-legalr.json · selected-contexts.zip
    ├── Scoring-Program-Task-LegalIR/     ← scoring.py chính thức của BTC
    ├── SourceCode/                       ← notebook này
    ├── Vòng 1 - Warm up/
    └── Vòng 2 - PublicTest/

[My Drive] DSC2026_Task1/artifacts/       ← notebook GHI vào đây (~1.5GB)
```

Artifact ghi vào **My Drive của bạn**, không ghi vào thư mục chia sẻ của nhóm — vừa
tránh làm rối thư mục chung, vừa tránh bị người khác xoá mất 1.5GB kết quả encode.

## ⚠ Bắt buộc làm 1 lần trước khi chạy

`drive.mount()` **chỉ mount My Drive**. Thư mục `UIT DS Challenge` đang ở
**"Shared with me"** nên Colab *không* thấy nó → phải tạo shortcut:

> drive.google.com → **Shared with me** → chuột phải **UIT DS Challenge**
> → **Organise** → **Add shortcut to Drive** → chọn **My Drive**

Bản này dùng **đường dẫn cố định**, không tự dò:

```
RAW = /content/drive/MyDrive/UIT DS Challenge/Task 1/LegalR - Public Test
```

Đặt dữ liệu ở chỗ khác thì sửa `RAW` ở cell Config. Cell **Giải nén
selected-contexts.zip** bung corpus vào `/content/scratch/contexts` rồi kiểm đủ
8.532 văn bản trước khi đi tiếp.

## Cách dùng
1. Làm bước shortcut ở trên.
2. Chạy tuần tự các Stage. Mỗi stage **cache artifact ra Drive** và tự bỏ qua nếu đã có
   → session Colab chết giữa đường thì chạy lại không mất việc đã làm.
3. Chạy trên **toàn bộ dữ liệu thật** (8.532 văn bản). Bước tốn thời gian duy nhất
   là Stage 5 encode corpus (~20 phút trên Kaggle 2×T4), và nó cache lại nên chỉ
   phải trả giá một lần.

## Mốc so sánh
Baseline TF-IDF toàn văn bản (truncate 20k ký tự), đo trên 500 câu train:

| | recall@5 | recall@20 | recall@100 |
|---|---|---|---|
| TF-IDF doc-level | **0.486** | 0.698 | 0.870 |

Khoảng cách `@100 = 0.87` vs `@5 = 0.486` cho thấy phần lớn điểm mất nằm ở **ranking**,
không phải retrieval → reranker là bước ăn điểm mạnh nhất.

## Ràng buộc cuộc thi (đọc từ `scoring.py` chính thức, không phải từ mô tả)
- **Tối đa 5 document_id / câu hỏi.** Vượt quá ở bất kỳ câu nào → câu đó nhận **0** cả
  Recall lẫn Precision.
- **Recall không cắt top-5** — code BTC chỉ *chặn* ở `len <= 5`, rồi lấy giao của
  **toàn bộ** danh sách nộp. Nên nộp **đúng 5 id** là tối ưu cho Recall.
- **Lệch số câu hoặc thiếu qid → `raise Exception` / `TypeError` → sập TOÀN BỘ bài nộp**,
  không phải chỉ mất điểm câu đó. Validator ở Stage 9 chặn cả hai.
- **id phải là string.** Code chấm dùng `set(truth) & set(pred)`; nộp `123` (int) sẽ
  không khớp `"123"` (str) → 0 điểm dù dự đoán đúng.
- **Recall là metric chính**, Precision chỉ dùng tie-break → **luôn nộp đủ 5 id**.
- **Tổng tham số toàn hệ thống < 4 tỷ** (gồm cả embedding layer).
- **Cấm mọi API.** Chỉ model open-source tự host.
- **Phải đăng ký model với BTC trước khi dùng** — xem cuối notebook.

## Model dùng trong notebook này

| Vai | Model | Tham số | Base | Context |
|---|---|---|---|---|
| bi-encoder | `AITeamVN/Vietnamese_Embedding_v2` | 0.6B | BAAI/bge-m3 | 2048 |
| reranker | `AITeamVN/Vietnamese_Reranker` | 0.6B | bge-reranker-v2-m3 | 2304 |
| | **Tổng** | **1.2B** < 4B | | |

### Vì sao bản này khác bản v1 chứ không chỉ đổi tên model

**1. Retriever chuyên tiếng Việt.** Vietnamese_Embedding_v2 dựa trên BGE-M3 và được
fine-tune trên lượng lớn triplet tiếng Việt.
Ít chunk hơn nhưng mỗi chunk giữ trọn ngữ cảnh một Điều → gộp `max` chunk→doc chính xác hơn.

**2. Reranker chuyên tiếng Việt.** `Vietnamese_Reranker` được huấn luyện trên khoảng
1,1 triệu triplet tiếng Việt và đã được đánh giá trên Legal Zalo 2021. Hiệu quả cuối cùng
vẫn phải được chọn bằng Recall@5 trên validation của chính cuộc thi.

**3. Chi phí vừa phải.** Retriever khoảng 0,3B ở 512 token nên phù hợp T4; reranker
0,6B chỉ chạy trên tập candidate. Corpus chỉ cần encode một lần và được cache theo model.

**4. Không dùng prefix E5.** Query và corpus được encode ở dạng text gốc.

**5. Fine-tune dùng GradCache.** `MultipleNegativesRankingLoss` phụ thuộc batch lớn,
nên Stage 8 giữ effective batch lớn nhưng chia thành minibatch nhỏ để vừa VRAM T4.
Stage 8 vì thế hạ `max_seq_length` xuống 512 khi train để đổi lấy batch lớn hơn. Nếu
kết quả zero-shot đã tốt, **cân nhắc bỏ hẳn fine-tune bi-encoder** cho bản này và dồn
công sức vào fine-tune reranker.

---

## Chạy ở đâu: Colab · Kaggle (2 GPU) · local

Notebook tự nhận môi trường ở Stage 0 (`ENV`) rồi tự chọn đường dẫn — không phải sửa gì.

| | Colab | Kaggle | local |
|---|---|---|---|
| đọc dữ liệu | My Drive (cần shortcut) | `/kaggle/input/<dataset>` (Add Input) | thư mục hiện tại |
| ghi artifact | `MyDrive/DSC2026_Task1/artifacts` | `/kaggle/working/DSC2026_Task1/artifacts` | `./work/artifacts` |
| đĩa tạm | `/content/scratch` | `/kaggle/temp/scratch` | `./scratch` |
| GPU | 1×T4 16GB | **2×T4 15GB** (hoặc 1×P100) | tuỳ máy |

### Kaggle: 3 việc phải làm

1. **Add Input** → Datasets → hai dataset đã ghim sẵn đường dẫn trong Stage 0:

   ```
   uit-task1-publictest  ->  .../uit-task1-publictest/LegalR - Public Test
   uit-task1-scoring     ->  .../uit-task1-scoring/Scoring-Program-Task-LegalIR
   ```

   Cả hai dạng `/kaggle/input/<slug>/...` và `/kaggle/input/datasets/<user>/<slug>/...`
   đều được nhận. Đổi tên dataset hay đổi tài khoản thì sửa `KAGGLE_DATA_HINTS` /
   `KAGGLE_SCORING_HINTS` ở Stage 0 — hoặc kệ, không trúng thì notebook tự dò
   trong `/kaggle/input`.

   Dataset scoring không bắt buộc, nhưng có thì Stage 3 sẽ đối chiếu hàm chấm điểm
   tự viết với `scoring.py` thật của BTC.
2. **Settings → Accelerator → GPU T4 ×2** (mặc định là None). Có 2 GPU thì Stage 5 và
   Stage 7 tự chia việc; 1 GPU vẫn chạy đúng, chỉ chậm hơn.
3. **Settings → Internet → On** — cần để tải trọng số model từ HuggingFace.

### Kaggle: giữ lại embedding giữa các session

Kaggle không có Drive, session chết là mất hết ngoài `/kaggle/working`. Vòng lặp đúng:

> Artifact Version 1 được đọc trực tiếp từ Kaggle Dataset `v3b-finetune-reranker-improve`;
> không cần attach Notebook Output.

Stage 0 dùng đường dẫn artifact cố định trong Kaggle Dataset (biến `ART_IN`) và Stage 5 đọc
lại từng shard embedding đã có ở đó → chỉ encode phần còn thiếu. Xem Stage 11 ở cuối.

Giới hạn cần biết: `/kaggle/working` **20GB**, session GPU **9h/tuần**, tối đa **12h/lần**.

### Cái gì thực sự chạy trên 2 GPU

| Stage | Cách dùng 2 GPU | Ghi chú |
|---|---|---|
| 5 · encode corpus | `start_multi_process_pool` — 1 process/GPU | bước nặng nhất, gần ×2 |
| 7 · rerank | 1 CrossEncoder/GPU, chia query theo thread | gần ×2 |
| 6 · dense search | không — 1 matmul, để trọn trên `cuda:0` | chia chỉ thêm đồng bộ |
| 8 · fine-tune | **nên chạy 1 GPU** | DataParallel cắt in-batch negatives của MNRL |


> ## 🔥 Bản B — fine-tune reranker
>
> Chạy được **song song** với bản A (`..._v3A_rerank_input.ipynb`) trong hai session Kaggle
> khác nhau. Bản này **encode lại corpus** (~20 phút) vì `MAX_CHUNKS_PER_DOC = 2500` →
> `_CAPTAG = '_m2500'` → cache mới, không đụng vào cache của bản A.
>
> Chi phí ~3–4h GPU. Thứ tự: Stage 0–7 → **7B.1, 7B.2** → **Stage 8B** (train) → 7B.3 + 7B.4
> với reranker mới → 9B + Stage 9.
>
> ✅ Bản production đặt **`SMOKE = False`**: train toàn bộ dữ liệu, không chạy thử 300 câu/30 bước.
>
> Bản B **không phụ thuộc kết quả của bản A**: nó train ở config `top30 × 8 chunk × maxlen
> 512` (giả thuyết "passage ngắn tập trung"). Bản A cho biết config nào tối ưu lúc
> *inference*; nếu A chỉ ra hình dạng khác thắng rõ, sửa `FT_RR_MAXLEN` / `RERANK_TOP` /
> `RERANK_CHUNKS_PER_DOC` rồi train lại — dữ liệu train sinh ở 8B.1 rẻ, chỉ model là đắt.


---
## Stage 0 — Config & môi trường

In [1]:
#@title Config
import os, sys, glob

# ---- Môi trường: Colab / Kaggle / local -------------------------------------
# Kiểm tra Kaggle TRƯỚC vì Kaggle không có /content, còn Colab thì luôn có.
# Trên Windows '/content' được hiểu là 'C:\content' nên phải chặn bằng os.name.
def _detect_env():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or os.environ.get('KAGGLE_URL_BASE'):
        return 'kaggle'
    if os.name == 'posix' and os.path.isdir('/kaggle/input'):
        return 'kaggle'
    if 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG'):
        return 'colab'
    if os.name == 'posix' and os.path.isdir('/content'):
        return 'colab'
    return 'local'

ENV       = _detect_env()
ON_COLAB  = ENV == 'colab'
ON_KAGGLE = ENV == 'kaggle'
PROFILE   = 'v3b_ft_reranker_improved'   # cache + tên file nộp riêng cho bản này.

# ---- Chạy Kaggle thành 2 Version --------------------------------------------
# Version 1: build cache + train reranker + lưu checkpoint, rồi kết thúc sạch.
# Version 2: attach output Version 1, load checkpoint và chỉ chạy val/private.
# Logic train/inference, seed và hyperparameter giữ nguyên; chỉ bỏ các bước đã xong.
RUN_PHASE = 2   #@param [1, 2] {type:'integer'}
assert RUN_PHASE in (1, 2), 'RUN_PHASE phải là 1 hoặc 2'
# CHẠY SONG SONG HAI BẢN: trên Kaggle mỗi notebook có /kaggle/working
# riêng nên an toàn. Trên Colab thì KHÔNG — cả hai mount cùng My Drive;
# phải để PROFILE khác nhau VÀ cap khác nhau (xem _CAPTAG dưới), nếu
# không hai session ghi đè shard .npy của nhau và làm hỏng cả hai.

# ---- GPU ---------------------------------------------------------------------
# '' = dùng hết GPU thấy được (Kaggle: 2×T4 -> Stage 5 và 7 chia việc cho cả hai).
#      Đây là giá trị nên dùng: Stage 8 tự ép mình về 1 GPU trong lúc train (xem
#      cell đó), nên không còn phải hy sinh tốc độ encode để fine-tune được nữa.
# '0' = buộc 1 GPU cho toàn bộ notebook (chỉ cần khi debug).
# Đổi biến này BẮT BUỘC phải Restart runtime rồi chạy lại từ đầu (torch đọc
# CUDA_VISIBLE_DEVICES một lần duy nhất lúc khởi tạo).
VISIBLE_GPUS = ''   #@param ['', '0', '1', '0,1'] {allow-input: true}
if VISIBLE_GPUS:
    os.environ['CUDA_VISIBLE_DEVICES'] = VISIBLE_GPUS

# ---- Nơi GHI artifact --------------------------------------------------------
# Colab : My Drive CỦA BẠN (không ghi vào thư mục chia sẻ của nhóm: ~1.5GB và
#         người khác có thể xoá mất).
# Kaggle: /kaggle/working — 20GB, và là thư mục DUY NHẤT còn lại sau "Save Version".
#         /kaggle/temp không vào output nên dùng làm đĩa tạm.
if ON_COLAB:
    ART_ROOT = '/content/drive/MyDrive/DSC2026_Task1'
    SCRATCH  = '/content/scratch'
elif ON_KAGGLE:
    ART_ROOT = '/kaggle/working/DSC2026_Task1'
    SCRATCH  = '/kaggle/temp/scratch'
else:
    ART_ROOT = './work'
    SCRATCH  = './scratch'
ART = f'{ART_ROOT}/artifacts'

# ---- Cache CHỈ ĐỌC từ Kaggle Dataset cố định -------------------------------
# Dataset này chứa artifact của Version 1; không dùng Notebook Output và không tự dò.
KAGGLE_ARTIFACTS = '/kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts'
ART_IN = [KAGGLE_ARTIFACTS] if ON_KAGGLE and os.path.isdir(KAGGLE_ARTIFACTS) else []

def cache_path(rel):
    """Tìm cache theo thứ tự: ART (ghi được) -> ART_IN (chỉ đọc). None nếu chưa có."""
    p = f'{ART}/{rel}'
    if os.path.exists(p):
        return p
    for d in ART_IN:
        q = f'{d}/{rel}'
        if os.path.exists(q):
            return q
    return None

def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

# ---- Dataset đã attach trên Kaggle -------------------------------------------
# Kaggle mount dataset theo /kaggle/input/<slug>, nhưng khi attach kèm tên chủ sở
# hữu thì đường dẫn thành /kaggle/input/datasets/<user>/<slug>. Ghim cả hai dạng;
# không trúng dạng nào thì cell sau tự dò trong /kaggle/input.
KAGGLE_DATA_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test',
    '/kaggle/input/uit-task1-publictest/LegalR - Public Test',
]
KAGGLE_TEST_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-private/private-official-legalr.json',
    '/kaggle/input/uit-task1-private/private-official-legalr.json',
]
KAGGLE_SCORING_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR',
    '/kaggle/input/uit-task1-scoring/Scoring-Program-Task-LegalIR',
]
# scoring.py THẬT của BTC -> Stage 3 dùng để đối chiếu hàm chấm điểm tự viết
SCORING_DIR = first_existing(KAGGLE_SCORING_HINTS + [
    './Scoring-Program-Task-LegalIR', '../Scoring-Program-Task-LegalIR'])

# ---- Nơi ĐỌC dữ liệu: đường dẫn cố định theo môi trường ---------------------
# Đường dẫn nào không tồn tại thì cell sau tự dò.
DATA_FILES = ('train.json',)
if ON_COLAB:
    RAW = '/content/drive/MyDrive/UIT DS Challenge/Task 1/LegalR - Public Test'
elif ON_KAGGLE:
    RAW = first_existing(KAGGLE_DATA_HINTS)
else:
    RAW = './LegalR - Public Test'
# File câu hỏi private tách khỏi RAW (RAW vẫn chứa train + corpus).
if ON_KAGGLE:
    TEST_PATH = first_existing(KAGGLE_TEST_HINTS)
else:
    TEST_PATH = f'{RAW}/private-official-legalr.json' if RAW else None
CTX_DIR = f'{SCRATCH}/contexts'   # nơi giải nén zip (đĩa tạm, KHÔNG vào Drive/output)

# ---- Chunking ----------------------------------------------------------------
CHUNK_CHARS        = 3000   # phù hợp context dài của Vietnamese_Embedding_v2
CHUNK_OVERLAP      = 500
MAX_CHUNKS_PER_DOC = 2500
# Đo trên dữ liệu thật: cap=150 cắt mất nội dung của 6.3% số lần gold doc
# xuất hiện, trong khi chỉ 41/8532 doc bị cap và bỏ cap chỉ thêm 3.5%
# chunk (154k -> 160k). Gold doc dài gấp ~3 lần trung bình corpus
# (p50 64k vs 22k ký tự) nên đây là mất mát thật.
#
# Cap PHẢI nằm trong tên cache. TAG/CHUNKS_REL bản v2 chỉ chứa
# CHUNK_CHARS, nên đổi cap mà không đổi tên là cache_path() trả về
# parquet + shard embedding CŨ trong im lặng. Giữ nguyên tên khi cap=150
# để cache của các lần chạy trước vẫn dùng lại được.
CHUNKER_VER = 't1'
# Đổi khi logic chunk_doc() ở Stage 2 đổi -> tên cache đổi theo. Thiếu cái này thì
# sửa chunker mà cache_path() vẫn trả về parquet + shard của chunker CŨ, im lặng.
_CAPTAG = (('' if MAX_CHUNKS_PER_DOC == 150 else f'_m{MAX_CHUNKS_PER_DOC}')
           + (f'_{CHUNKER_VER}' if CHUNKER_VER else ''))

# ---- Retrieval ---------------------------------------------------------------
BM25_K1, BM25_B = 0.9, 0.4  # tuned cho document dài
CAND_CHUNKS     = 1000      # số chunk lấy ra trước khi gộp về document
CAND_DOCS       = 100       # số doc mỗi nhánh đưa vào RRF
RRF_K           = 60
RERANK_TOP      = 30       # số doc đưa vào cross-encoder
TOPK_SUBMIT     = 5         # RÀNG BUỘC CỨNG của BTC

# Cách dùng điểm của cross-encoder. ĐỪNG đoán — Stage 7 in cả ba, chọn theo val:
#   'rrf'     : RRF giữa thứ tự hybrid và thứ tự rerank. Reranker sai một câu thì
#               phiếu của retriever vẫn giữ được gold trong top-5 -> an toàn nhất.
#   'replace' : tin hẳn reranker (cách cũ). Chỉ chọn khi val cho thấy nó thắng rõ.
#   'none'    : bỏ rerank. Đúng là một lựa chọn: ở lần chạy thử trên corpus rút
#               gọn, hybrid RRF (recall@5 0.919) còn cao hơn 'replace' (0.858).
RERANK_MODE     = 'rrf'   #@param ['rrf', 'replace', 'none']

# ---- Ba tham số thêm sau khi đọc log profile v2 ------------------------------
# Log v2 nói rất rõ:
#   dense              recall@5 = 0.8996  <- TỐT NHẤT
#   hybrid RRF         recall@5 = 0.8699  <- tụt 3 điểm so với dense
#   hybrid + rerank    recall@5 = 0.8973  <- vẫn chưa vượt nổi dense đơn thuần
# Nguyên nhân: RRF cho BM25 (recall@5 = 0.6868) một phiếu NGANG BẰNG dense.
# Và dense recall@100 = 0.9897 -> khâu tìm ứng viên hết chỗ cải thiện; toàn bộ
# dư địa còn lại nằm ở XẾP HẠNG trong top-30 (trần ~0.975).

# Base đưa vào cross-encoder và dùng cho submission.
# Stage 7 đo CẢ HAI rồi in ra cái nào thắng — sửa biến này theo bảng đó.
RERANK_BASE = 'dense'   #@param ['dense', 'hybrid']

# Trọng số RRF cho (bm25, dense). (1,1) là RRF cổ điển, chỉ đúng khi hai kênh
# mạnh ngang nhau. Stage 6 sweep vài giá trị rồi chọn theo val.
RRF_WEIGHTS = (1.0, 3.0)

# Số chunk mỗi doc đưa cho cross-encoder; điểm doc = MAX trên các chunk đó.
# Trước đây = 1: với nhiều chunk/doc, reranker chỉ
# được đọc 1/150 nội dung của doc -> chọn nhầm chunk đại diện là gold chết oan.
# Chi phí GPU tăng tuyến tính (30 doc x 3 chunk = 90 cặp/query).
RERANK_CHUNKS_PER_DOC = 8   # bản B: cố định = config lúc train reranker

# ---- Models (phải đăng ký với BTC trước khi dùng) ----------------------------
BIENCODER = 'AITeamVN/Vietnamese_Embedding_v2'    # 0.6B
RERANKER  = 'AITeamVN/Vietnamese_Reranker'        # 0.6B -> tổng 1.2B < 4B
MAX_SEQ   = 1024
def format_dense_queries(texts):
    return list(texts)
def format_dense_passages(texts):
    return list(texts)
FT_BIENCODER_REL = f'models/biencoder-ft-{PROFILE}'
if RUN_PHASE == 1:
    FT_BIENCODER_PATH = f'{ART}/{FT_BIENCODER_REL}'
else:
    FT_BIENCODER_PATH = cache_path(FT_BIENCODER_REL)
    assert FT_BIENCODER_PATH and os.path.isdir(FT_BIENCODER_PATH), (
        'Không thấy bi-encoder đã fine-tune trong output Version 1. '
        f'Không thấy checkpoint trong Kaggle Dataset artifact: {KAGGLE_ARTIFACTS}')
# Stage 5 encode bằng checkpoint nào lúc BẮT ĐẦU chạy notebook.
#   False -> checkpoint gốc. Đây là giá trị đúng cho lần chạy đầu: Stage 8 cần
#            retrieval của model gốc để mine hard negative.
#   Stage 8 train xong sẽ TỰ bật biến này lên True + encode lại toàn corpus,
#   nên không cần sửa tay rồi chạy lại notebook từ đầu nữa.
USE_FINETUNED = (RUN_PHASE == 2)
DO_FINETUNE   = (RUN_PHASE == 1)  # train đúng một lần ở Version 1
# TẮT ở cả hai bản v3. recall@100 = 0.9897: retriever đã xong việc, dư địa
# nằm ở XẾP HẠNG. Fine-tune bi-encoder tốn thêm một lần encode lại toàn
# corpus mà tối ưu đúng cái đã hết chỗ tối ưu.

# ---- Eval --------------------------------------------------------------------
N_VAL = 1000
SEED  = 42

# ---- Batch: các số này là batch cho MỖI GPU ---------------------------------
# T4 Colab 16GB và T4 Kaggle 15GB gần như bằng nhau -> giữ nguyên batch, tốc độ
# tăng là do chia việc cho 2 GPU chứ không phải do batch to hơn. Kaggle P100
# (1×16GB) cũng chạy đúng cấu hình này.
ENC_SHARD     = 20000   # encode theo shard, checkpoint từng shard ra đĩa
ENC_BATCH     = 16
RERANK_MAXLEN = 512      # bản B: passage ngắn tập trung, = maxlen lúc train
RERANK_BATCH  = 8
RERANK_FP16   = False    # V11 đo fp32 để tránh score saturation ở reranker

print('ENV =', ENV, '| PROFILE =', PROFILE, '| RUN_PHASE =', RUN_PHASE, '| RERANK_MODE =', RERANK_MODE)
print('ART      =', ART)
print('SCRATCH  =', SCRATCH)
if RAW:
    print('RAW      =', RAW)
if TEST_PATH:
    print('PRIVATE  =', TEST_PATH)
if SCORING_DIR:
    print('scoring.py BTC:', SCORING_DIR)
if ART_IN:
    print('cache chỉ đọc:', ART_IN)


ENV = kaggle | PROFILE = v3b_ft_reranker_improved | RUN_PHASE = 2 | RERANK_MODE = rrf
ART      = /kaggle/working/DSC2026_Task1/artifacts
SCRATCH  = /kaggle/temp/scratch
RAW      = /kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test
PRIVATE  = /kaggle/input/datasets/kitnehi1211/uit-task1-private/private-official-legalr.json
scoring.py BTC: /kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR
cache chỉ đọc: ['/kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts']


In [2]:
#@title Mount Drive (Colab) + cài dependency + liệt kê GPU
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Pin cùng một bản Sentence Transformers trên cả Colab lẫn Kaggle.
ST_WANT = '5.4.0'  # tương thích cả legal E5 và Vietnamese cross-encoder
try:
    import sentence_transformers as _st
    ST_HAVE = _st.__version__
except ImportError:
    ST_HAVE = None

if ST_HAVE != ST_WANT:
    !pip -q install sentence-transformers=={ST_WANT} pyarrow py_vncorenlp 2>&1 | tail -1
    import importlib, sentence_transformers as _st
    ST_HAVE = importlib.reload(_st).__version__

print('sentence-transformers', ST_HAVE, '(bản đã test:', ST_WANT + ')')

import os
for d in (ART, f'{ART}/emb', f'{ART}/models', SCRATCH):
    os.makedirs(d, exist_ok=True)

import torch
N_GPU   = torch.cuda.device_count()
DEVICES = [f'cuda:{i}' for i in range(N_GPU)] or ['cpu']
DEV     = DEVICES[0]

print('torch', torch.__version__, '| GPU thấy được:', N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB')
if N_GPU > 1:
    print(f'-> Stage 5 (encode corpus) và Stage 7 (rerank) sẽ chia việc cho {N_GPU} GPU')
    print('   Stage 8 (fine-tune) tự ép về 1 GPU trong lúc train, không cần sửa gì.')
elif N_GPU == 1:
    print('-> 1 GPU: pipeline chạy tuần tự như bình thường')
else:
    print('-> KHÔNG có GPU. Stage 1-4 (BM25) vẫn chạy được và nên chạy ở session CPU '
          'để tiết kiệm quota; Stage 5 trở đi bắt buộc phải có GPU.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 14.5 MB/s eta 0:00:00
sentence-transformers 5.4.0 (bản đã test: 5.4.0)
torch 2.10.0+cu128 | GPU thấy được: 2
  cuda:0  Tesla T4  15.6 GB
  cuda:1  Tesla T4  15.6 GB
-> Stage 5 (encode corpus) và Stage 7 (rerank) sẽ chia việc cho 2 GPU
   Stage 8 (fine-tune) tự ép về 1 GPU trong lúc train, không cần sửa gì.


In [3]:
#@title Dò dữ liệu + giải nén selected-contexts.zip
# Ưu tiên RAW cố định ở Config; không có thì dò (Kaggle: /kaggle/input/<dataset>).
# Giải nén vào đĩa tạm chứ KHÔNG vào Drive / /kaggle/working: 8.532 file nhỏ ghi
# lên Drive mất hàng giờ, và đếm vào giới hạn 20GB output của Kaggle.
import os, glob, zipfile, time

def find_data_dir(roots, max_depth=6):
    """Thư mục đầu tiên chứa đủ DATA_FILES."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        base = root.rstrip('/').count('/')
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count('/') - base >= max_depth:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames
                           if not d.startswith('.') and d != '__MACOSX']
            if all(f in filenames for f in DATA_FILES):
                return dirpath
    return None

def data_roots():
    if ON_COLAB:
        return ['/content/drive/MyDrive',
                '/content/drive/.shortcut-targets-by-id',   # nơi shortcut trỏ tới
                '/content/drive/Shareddrives',
                '/content']
    if ON_KAGGLE:
        return ['/kaggle/input']                            # mọi dataset đã attach
    return ['.', '..']

def n_ctx(d):
    return len(glob.glob(f'{d}/**/context_*.json', recursive=True)) if d else 0

if RAW is None or not all(os.path.exists(f'{RAW}/{f}') for f in DATA_FILES):
    print('đường dẫn cố định không dùng được -> đang dò...', flush=True)
    RAW = find_data_dir(data_roots())

assert RAW, (
    'KHÔNG tìm thấy dữ liệu train/corpus (cần train.json và selected-contexts).\n\n'
    'Colab: thư mục "UIT DS Challenge" thường ở "Shared with me" mà Colab không\n'
    '  mount phần đó. Sửa 1 lần: drive.google.com -> Shared with me -> chuột phải\n'
    '  "UIT DS Challenge" -> Organise -> Add shortcut to Drive -> My Drive.\n\n'
    'Kaggle: phải Add Input dataset chứa dữ liệu Task 1 vào notebook\n'
    '  (Notebook -> Add Input -> Datasets). Kiểm tra: !ls /kaggle/input\n\n'
    'Hoặc gán tay: RAW = "<đường dẫn tới thư mục LegalR - Public Test>"')

print('RAW =', RAW)

if TEST_PATH is None or not os.path.isfile(TEST_PATH):
    matches = []
    for root in data_roots():
        if os.path.isdir(root):
            matches.extend(glob.glob(f'{root}/**/private-official-legalr.json', recursive=True))
    TEST_PATH = matches[0] if matches else None
assert TEST_PATH and os.path.isfile(TEST_PATH), (
    'Không thấy private-official-legalr.json. Trên Kaggle hãy Add Input dataset '
    'kitnehi1211/uit-task1-private.')
print('PRIVATE =', TEST_PATH)

CTX_ZIP = f'{RAW}/selected-contexts.zip'

if cache_path('corpus.parquet'):
    print('corpus.parquet đã có -> bỏ qua giải nén')
elif n_ctx(RAW):
    CTX_DIR = RAW                     # dataset đã bung sẵn (hay gặp trên Kaggle)
    print(f'đã bung sẵn trong RAW: {n_ctx(CTX_DIR)} file')
elif n_ctx(CTX_DIR):
    print(f'đã giải nén trước đó: {n_ctx(CTX_DIR)} file')
else:
    assert os.path.exists(CTX_ZIP), (
        f'Không thấy {CTX_ZIP} và cũng không thấy file context_*.json nào.\n'
        'Colab: kiểm tra shortcut "UIT DS Challenge" trong My Drive.\n'
        'Kaggle: dataset đã attach phải chứa selected-contexts (zip hoặc đã bung).')
    os.makedirs(CTX_DIR, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(CTX_ZIP) as z:
        print(f'{len(z.namelist())} entry, đang giải nén...', flush=True)
        z.extractall(CTX_DIR)
    print(f'xong {time.time()-t0:.0f}s -> {n_ctx(CTX_DIR)} file context')

if not cache_path('corpus.parquet'):
    n = n_ctx(CTX_DIR)
    assert n == 8532, f'mong đợi 8532 file context, thấy {n} — zip/dataset có thể thiếu'
    print('OK: đủ 8532 văn bản')


RAW = /kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test
PRIVATE = /kaggle/input/datasets/kitnehi1211/uit-task1-private/private-official-legalr.json
corpus.parquet đã có -> bỏ qua giải nén


---
## Stage 1 — Build corpus

Đọc 8.532 file `context_*.json` → **một** file parquet.

Hai vấn đề dữ liệu được xử lý ở đây:
- Zip giải nén **lồng hai lớp** (`selected-contexts/selected-contexts/...`) → glob đệ quy trong `CTX_DIR`.
- **1.125/8.532 văn bản (13%) thiếu field `name`** → lấy lại tiêu đề từ slug trong `link`.
  Tiêu đề là tín hiệu rất mạnh (chứa loại văn bản + chủ đề + năm), bỏ 13% là mất điểm thật.

Cell này chỉ đọc `CTX_DIR` (đã bung ở cell Giải nén phía trên) — không tự dò thư mục khác.

In [4]:
#@title Build corpus.parquet
import json, re, glob, os, time
import pandas as pd

CORPUS_PQ = f'{ART}/corpus.parquet'

def _title_from_link(link):
    slug = (link or '').rstrip('/').rsplit('/', 1)[-1]
    slug = re.sub(r'\.aspx$', '', slug, flags=re.I)
    return slug.replace('-', ' ').strip()

def build_corpus():
    # Chỉ đọc CTX_DIR — cell "Giải nén selected-contexts.zip" ở trên đã bung sẵn.
    # Zip lồng hai lớp (selected-contexts/selected-contexts/...) nên glob đệ quy.
    files = sorted(glob.glob(f'{CTX_DIR}/**/context_*.json', recursive=True))
    assert files, f'{CTX_DIR} trống — hãy chạy lại cell "Giải nén selected-contexts.zip".'
    print(f'{len(files)} file context', flush=True)

    rows, n_noname = [], 0
    for i, fp in enumerate(files):
        with open(fp, encoding='utf-8') as f:
            d = json.load(f)
        title = (d.get('name') or '').replace('-', ' ').strip()
        if not title:
            n_noname += 1
            title = _title_from_link(d.get('link', ''))
        # gộp mọi whitespace: passage có rất nhiều khoảng trắng / newline rác.
        # Làm sớm để regex tách 'Điều N' ở Stage 2 hoạt động ổn định.
        passage = re.sub(r'\s+', ' ', d.get('passage') or '').strip()
        rows.append({'doc_id': str(d['id']), 'title': title,
                     'link': d.get('link', ''), 'passage': passage})
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(files)}', flush=True)

    df = pd.DataFrame(rows).drop_duplicates('doc_id').reset_index(drop=True)
    print(f'-> {len(df)} doc | thiếu name (đã vá từ link): {n_noname}')
    df.to_parquet(CORPUS_PQ, index=False)
    return df

t0 = time.time()
_pq = cache_path('corpus.parquet')      # ART trước, rồi tới dataset chỉ đọc (Kaggle)
if _pq:
    corpus = pd.read_parquet(_pq)
    print(f'load cache: {len(corpus)} doc <- {_pq}')
else:
    corpus = build_corpus()

print('%.1fs' % (time.time() - t0))
print('passage chars: mean %d | p50 %d | max %d' % (
    corpus.passage.str.len().mean(), corpus.passage.str.len().median(),
    corpus.passage.str.len().max()))
corpus.head(3)[['doc_id', 'title']]

load cache: 8532 doc <- /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/corpus.parquet
7.0s
passage chars: mean 39451 | p50 22089 | max 5787180


,doc_id,title
0,100050,TCVN 13268 1 2021 Bao ve thuc vat dieu tra sin...
1,100062,Thong tu 17 2022 TT BGTVT sua doi Thong tu 12 ...
2,100109,Quyet dinh 569 QD TTg 2022 Chien luoc phat tri...


In [5]:
#@title Load train / private + val split cố định
import json, random

with open(f'{RAW}/train.json', encoding='utf-8') as f:
    train_raw = json.load(f)
with open(TEST_PATH, encoding='utf-8') as f:
    private_raw = json.load(f)

# answer có thể là int hoặc str tuỳ file -> luôn ép str
QUESTIONS = {qid: v['question'] for qid, v in train_raw.items()}
GOLD = {qid: set(map(str, v['answer'])) for qid, v in train_raw.items()}
PRIVATE = {qid: v['question'] for qid, v in private_raw.items()}

# Split cố định theo seed + sorted key -> mọi session cho cùng một val set.
# Đây là điều kiện bắt buộc để so sánh các thí nghiệm với nhau.
all_qids = sorted(train_raw.keys())
random.Random(SEED).shuffle(all_qids)
VAL_QIDS   = all_qids[:N_VAL]
TRAIN_QIDS = all_qids[N_VAL:]

print(f'train {len(TRAIN_QIDS)} | val {len(VAL_QIDS)} | private {len(PRIVATE)}')

# kiểm tra gold có nằm trong corpus không
corpus_ids = set(corpus.doc_id)
missing = {g for q in all_qids for g in GOLD[q]} - corpus_ids
print('gold doc thiếu trong corpus:', len(missing))

train 6000 | val 1000 | private 2080
gold doc thiếu trong corpus: 0


---
## Stage 2 — Chunking theo cấu trúc pháp quy

Passage trung bình **41.000 ký tự** mà bi-encoder chỉ nhận ~256 token (~1.000 ký tự).
Không chunk là không dùng được dense retrieval.

Ba quyết định thiết kế ở đây:
- **Cắt theo `Điều N`** thay vì cắt cứng theo độ dài — ranh giới ngữ nghĩa của văn bản
  pháp luật nằm đúng ở đó. Điều ngắn thì gộp lại, Điều dài thì cắt tiếp có overlap.
- **Prepend tiêu đề vào mọi chunk.** Câu hỏi rất ngắn (trung bình 20 từ) nên tiêu đề
  văn bản là tín hiệu định chủ đề cực mạnh.
- **Chunk đầu tiên = tiêu đề + phần mở đầu**, nơi chứa số hiệu, cơ quan ban hành, trích yếu.
- **Cap `MAX_CHUNKS_PER_DOC`**: nếu không, vài văn bản 6M ký tự sẽ chiếm phần lớn index
  và bóp méo điểm `max` khi gộp chunk→doc.

In [6]:
#@title Build chunks.parquet
import re, os, time
import pandas as pd

CHUNKS_REL = f'chunks_c{CHUNK_CHARS}{_CAPTAG}.parquet'
CHUNKS_PQ  = f'{ART}/{CHUNKS_REL}'

DIEU_RE = re.compile(r'(?=Điều\s+\d+\s*[\.\:])')

def _split_long(s, size, overlap):
    """Cắt s thành các đoạn <= size ký tự, ưu tiên cắt ở ranh giới câu."""
    out, i = [], 0
    while i < len(s):
        j = min(i + size, len(s))
        if j < len(s):
            # lùi về dấu chấm gần nhất trong 200 ký tự cuối cho khỏi cắt giữa câu
            cut = s.rfind('. ', i + size - 200, j)
            if cut > i:
                j = cut + 1
        out.append(s[i:j].strip())
        if j >= len(s):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if c]

def chunk_doc(title, passage):
    """-> list các đoạn text (chưa gắn tiêu đề)."""
    pieces = [p.strip() for p in DIEU_RE.split(passage) if p.strip()]
    if not pieces:
        pieces = [passage]

    merged = []
    buf = ''
    for p in pieces:
        if len(p) > CHUNK_CHARS:
            if buf:
                merged.append(buf); buf = ''
            merged.extend(_split_long(p, CHUNK_CHARS, CHUNK_OVERLAP))
        elif len(buf) + len(p) + 1 <= CHUNK_CHARS:
            buf = f'{buf} {p}'.strip()
        else:
            merged.append(buf); buf = p
    if buf:
        merged.append(buf)

    # chunk mở đầu: chứa số hiệu / cơ quan ban hành / trích yếu
    head = passage[:CHUNK_CHARS].strip()
    if head and (not merged or merged[0][:200] != head[:200]):
        merged.insert(0, head)
    return merged[:MAX_CHUNKS_PER_DOC]

def build_chunks(corpus):
    rows, n_titleonly = [], 0
    for n, r in enumerate(corpus.itertuples(index=False)):
        cs = chunk_doc(r.title, r.passage)
        if not cs:
            # 20 văn bản có `passage` RỖNG và cũng không có field `name` (tiêu đề
            # được vá từ slug trong `link` ở trên). chunk_doc() chỉ chunk passage
            # nên chúng nhận 0 chunk -> dense retrieval KHÔNG BAO GIỜ truy hồi
            # được. 6/20 doc này từng là gold, và 9 câu hỏi train mất trắng vì mọi
            # gold của chúng nằm trong nhóm đó. Phát một chunk chỉ chứa tiêu đề:
            # tiêu đề vá từ slug link có loại văn bản + chủ đề + năm, đủ để khớp ở
            # mức chủ đề — hơn hẳn không có gì.
            cs, n_titleonly = [''], n_titleonly + 1
        for c in cs:
            # tiêu đề đi kèm mọi chunk -> chunk lẻ vẫn biết mình thuộc văn bản nào
            rows.append((r.doc_id, f'{r.title}. {c}'.strip()))
        if (n + 1) % 1000 == 0:
            print(f'  {n+1}/{len(corpus)} doc -> {len(rows)} chunk', flush=True)
    if n_titleonly:
        print(f'-> {n_titleonly} doc passage rỗng: đã phát chunk chỉ-tiêu-đề')
    df = pd.DataFrame(rows, columns=['doc_id', 'text'])
    df.to_parquet(CHUNKS_PQ, index=False)
    return df

t0 = time.time()
_cpq = cache_path(CHUNKS_REL)
if _cpq:
    chunks = pd.read_parquet(_cpq)
    print('load cache <-', _cpq)
else:
    chunks = build_chunks(corpus)
print('%d chunk | %.1fs' % (len(chunks), time.time() - t0))

# map chunk -> doc dạng số nguyên, dùng suốt phần sau
DOC_IDS = corpus.doc_id.tolist()
DOC2IX  = {d: i for i, d in enumerate(DOC_IDS)}
import numpy as np
CHUNK2DOC = np.array([DOC2IX[d] for d in chunks.doc_id], dtype=np.int32)
print('chunk/doc trung bình: %.1f' % (len(chunks) / len(corpus)))
chunks.text.str.len().describe()[['mean', '50%', 'max']]

load cache <- /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/chunks_c3000_m2500_t1.parquet
151034 chunk | 7.6s
chunk/doc trung bình: 17.7


mean    2537.283923
50%     2912.000000
max     3147.000000
Name: text, dtype: float64

---
## Stage 3 — Hạ tầng đánh giá

**Làm cái này trước khi tối ưu bất cứ gì.** `private-official-legalr.json` không có đáp án và chỉ
được nộp 10 lần/ngày, nên mọi quyết định kỹ thuật phải ra từ val set này — miễn phí, không
giới hạn số lần thử.

Công thức theo đúng BTC: `Recall = mean_q |gold ∩ pred| / |gold|` với `pred` cắt ở 5 doc.

Luôn in cả `@5`, `@20`, `@100` vì chúng chẩn đoán hai bệnh khác nhau:
- `@100` thấp → **retriever** yếu (văn bản đúng không vào nổi danh sách ứng viên)
- `@100` cao mà `@5` thấp → **ranking** yếu (tìm ra rồi nhưng không đẩy lên top)

In [7]:
#@title Metrics + experiment log
import numpy as np, pandas as pd, os, datetime

def official_score(sub, truth):
    """Bản sao CHÍNH XÁC của scoring.py mà BTC dùng trên Codabench.

    sub: {qid: {'answer': [id]}} — đúng định dạng file nộp
    truth: {qid: [id]}

    Ba chi tiết trong code của BTC mà đọc mô tả không thấy:
    1. Recall KHÔNG cắt top-5. Nó chỉ *chặn* ở len<=5: quá 5 id -> câu đó 0 điểm.
       Nộp đúng 5 id là tối ưu cho Recall.
    2. Số lượng key phải khớp tuyệt đối, lệch là raise Exception -> TOÀN BỘ bài nộp
       lỗi, không phải chỉ mất điểm câu đó.
    3. Thiếu key -> len(None) -> TypeError -> cũng sập toàn bài.
    """
    y_pred = {k: v['answer'] for k, v in sub.items()}
    if len(y_pred) != len(truth):
        raise Exception('Samples in predict not match with reference')
    ok = lambda k: 0 < len(y_pred[k]) <= TOPK_SUBMIT
    recall = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(truth[k])
                      if ok(k) else 0 for k in truth])
    precision = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(y_pred[k])
                         if ok(k) else 0 for k in y_pred])
    return {'recall': float(recall), 'precision': float(precision)}

def evaluate(preds, gold, ks=(5, 10, 20, 30, 50, 100)):
    """preds: {qid: [doc_id đã xếp hạng]} -> recall@k chẩn đoán + điểm chính thức.

    recall@20/@100 KHÔNG phải metric của cuộc thi — chúng để chẩn đoán:
      @100 thấp        -> retriever yếu (văn bản đúng không vào nổi danh sách)
      @100 cao, @5 thấp -> ranking yếu (tìm ra rồi mà không đẩy lên top)

    recall@RERANK_TOP (mặc định @30) là **trần cứng** của Stage 7: cross-encoder
    chỉ được nhìn top-RERANK_TOP, doc không lọt vào đó thì không cách nào cứu.
    Đọc `@30` và `@5` cạnh nhau -> biết ngay còn bao nhiêu điểm đang bỏ trên bàn.
    """
    out = {}
    for k in ks:
        out[f'recall@{k}'] = float(np.mean([
            len(gold[q] & set(preds.get(q, [])[:k])) / len(gold[q]) for q in gold]))
    # điểm chính thức: dựng đúng shape file nộp rồi chấm bằng code của BTC
    sub = {q: {'answer': list(preds.get(q, [])[:TOPK_SUBMIT])} for q in gold}
    off = official_score(sub, {q: list(gold[q]) for q in gold})
    out['OFFICIAL_recall'] = off['recall']
    out['OFFICIAL_precision'] = off['precision']
    return out

def show(name, preds, gold=None, qids=None):
    gold = gold or {q: GOLD[q] for q in (qids or VAL_QIDS)}
    m = evaluate(preds, gold)
    print(name.ljust(28), ' '.join(f'{k}={v:.4f}' for k, v in m.items()))
    return m

LOG_CSV = f'{ART}/experiments.csv'

def log_exp(name, metrics, note=''):
    """Ghi lại mọi thí nghiệm. Bảng này là thứ bạn cần để chọn cấu hình cho private
    test và để viết paper nếu vào top 10 — đừng bỏ qua."""
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'),
           'profile': PROFILE, 'run': name, 'note': note, **metrics}
    df = pd.DataFrame([row])
    if os.path.exists(LOG_CSV):
        df = pd.concat([pd.read_csv(LOG_CSV), df], ignore_index=True)
    df.to_csv(LOG_CSV, index=False)
    return df.tail(5)

print('ok')


# ---- Đối chiếu official_score() với scoring.py THẬT của BTC ------------------
# Dataset uit-task1-scoring có sẵn mã chấm điểm Codabench -> nạp trực tiếp
# eval_retrieval() của BTC và so với bản chép tay ở trên trên vài trường hợp
# biên. Nếu lệch thì mọi con số OFFICIAL_* trong notebook đều vô nghĩa.
import importlib.util, glob as _glob

def _load_btc_eval():
    cands = ([f'{SCORING_DIR}/scoring.py'] if SCORING_DIR else []) + \
            sorted(_glob.glob('/kaggle/input/**/Scoring-Program-Task-LegalIR/scoring.py',
                              recursive=True))
    for p in cands:
        if not os.path.exists(p):
            continue
        spec = importlib.util.spec_from_file_location('btc_scoring', p)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)          # main() có guard __main__ -> import an toàn
        return mod.eval_retrieval, p
    return None, None

_btc_eval, _btc_path = _load_btc_eval()
if _btc_eval is None:
    print('không thấy scoring.py của BTC -> bỏ qua đối chiếu '
          '(Kaggle: Add Input dataset uit-task1-scoring)')
else:
    _truth = {'a': ['1'], 'b': ['2', '3'], 'c': ['4']}
    _cases = {
        'đúng 1/1':        {'a': ['1'], 'b': ['2', '3'], 'c': ['4']},
        'nộp 5 id':        {'a': ['1', 'x', 'y', 'z', 'w'], 'b': ['2', '3', 'x', 'y', 'z'],
                            'c': ['q', 'w', 'e', 'r', 't']},
        'nộp 6 id -> 0':   {'a': ['1', 'x', 'y', 'z', 'w', 'v'], 'b': ['2'], 'c': ['4']},
    }
    _ok = True
    for _name, _p in _cases.items():
        _mine = official_score({k: {'answer': v} for k, v in _p.items()}, _truth)
        _theirs = _btc_eval({k: {'answer': v} for k, v in _p.items()}, _truth)
        _same = (abs(_mine['recall'] - _theirs['recall']) < 1e-9 and
                 abs(_mine['precision'] - _theirs['precision']) < 1e-9)
        _ok &= _same
        print(f'  {_name:16s} recall={_theirs["recall"]:.4f} '
              f'precision={_theirs["precision"]:.4f}  {"khớp" if _same else "LỆCH!"}')
    assert _ok, 'official_score() lệch so với scoring.py của BTC — sửa trước khi tin số liệu'
    print('official_score() khớp scoring.py của BTC <-', _btc_path)


ok
  đúng 1/1         recall=1.0000 precision=1.0000  khớp
  nộp 5 id         recall=0.6667 precision=0.2000  khớp
  nộp 6 id -> 0    recall=0.5000 precision=0.6667  khớp
official_score() khớp scoring.py của BTC <- /kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR/scoring.py


In [8]:
#@title Gộp chunk -> doc bằng MAX, rồi lấy top-N doc
def chunks_to_docs(chunk_scores, chunk_top_ix, topn=CAND_DOCS):
    """Điểm doc = MAX điểm các chunk của nó (không phải mean).

    Gold là "văn bản CHỨA thông tin cần thiết" -> một chunk khớp là đủ. Lấy mean sẽ
    phạt oan các văn bản dài mà chỉ một Điều liên quan.
    """
    best = {}
    for ix, s in zip(chunk_top_ix, chunk_scores):
        d = CHUNK2DOC[ix]
        if s > best.get(d, -1e30):
            best[d] = s
    ranked = sorted(best.items(), key=lambda x: -x[1])[:topn]
    return [DOC_IDS[d] for d, _ in ranked], [s for _, s in ranked]

print('ok')

ok


---
## Stage 4 — BM25 (chunk-level)

Không cần GPU — **chạy stage này trong session CPU** để tiết kiệm quota GPU, đó là tài
nguyên khan hiếm nhất của bạn.

Cài đặt BM25 dạng sparse matrix (`W @ q`) thay vì `rank_bm25`: với ~500k chunk thì
`rank_bm25` chậm tới mức không dùng được, còn sparse matmul chấm 1 query mất vài chục ms.

BM25 bù đúng chỗ dense yếu: thuật ngữ pháp lý hiếm (*"Kiểm ngư viên trung cấp"*,
*"hành lang an toàn kỹ thuật anten"*) mà embedding thường trượt hẳn.

In [9]:
#@title BM25 sparse
import numpy as np, scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

class SparseBM25:
    """BM25 Okapi tiền tính thành ma trận trọng số -> truy vấn = 1 phép matmul sparse."""

    def __init__(self, k1=BM25_K1, b=BM25_B, min_df=3, max_features=400_000):
        self.k1, self.b = k1, b
        self.vec = CountVectorizer(lowercase=True, min_df=min_df,
                                   max_features=max_features, dtype=np.int32)

    def fit(self, texts):
        X = self.vec.fit_transform(texts).tocsc()
        n, _ = X.shape
        dl = np.asarray(X.sum(axis=1)).ravel()
        avgdl = dl.mean()
        df = np.diff(X.indptr)
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)

        X = X.tocoo()
        tf = X.data.astype(np.float32)
        denom = tf + self.k1 * (1 - self.b + self.b * dl[X.row] / avgdl)
        data = idf[X.col] * tf * (self.k1 + 1) / denom
        self.W = sp.csr_matrix((data, (X.row, X.col)), shape=(n, len(idf),))
        self.W = self.W.T.tocsr()      # (vocab, n_chunk) -> query nhanh hơn
        return self

    def search(self, queries, topk=CAND_CHUNKS):
        Q = self.vec.transform(queries)
        Q.data[:] = 1.0               # query dùng binary term presence
        S = (Q.astype(np.float32) @ self.W).toarray()   # (nq, n_chunk)
        k = min(topk, S.shape[1])
        ix = np.argpartition(-S, k - 1, axis=1)[:, :k]
        srt = np.take_along_axis(S, ix, 1).argsort(axis=1)[:, ::-1]
        ix = np.take_along_axis(ix, srt, 1)
        return ix, np.take_along_axis(S, ix, 1)

import time, pickle
t0 = time.time()
bm25 = SparseBM25().fit(chunks.text.tolist())
print('BM25 fit: %.1fs | vocab %d' % (time.time() - t0, bm25.W.shape[0]))

BM25 fit: 49.0s | vocab 48842


In [10]:
#@title Đánh giá BM25 trên val
def retrieve_bm25(qids, qtext, batch=64):
    preds = {}
    for i in range(0, len(qids), batch):
        bq, bt = qids[i:i+batch], qtext[i:i+batch]
        ix, sc = bm25.search(bt)
        for q, row_ix, row_sc in zip(bq, ix, sc):
            preds[q] = chunks_to_docs(row_sc, row_ix)[0]
    return preds

t0 = time.time()
bm25_val = retrieve_bm25(VAL_QIDS, [QUESTIONS[q] for q in VAL_QIDS])
print('%.1fs' % (time.time() - t0))
m = show('BM25 chunk-level', bm25_val)
log_exp('bm25', m, f'chunk={CHUNK_CHARS} k1={BM25_K1} b={BM25_B}')

8.7s
BM25 chunk-level             recall@5=0.6917 recall@10=0.7903 recall@20=0.8597 recall@30=0.8937 recall@50=0.9179 recall@100=0.9484 OFFICIAL_recall=0.6917 OFFICIAL_precision=0.1486


,time,profile,run,note,recall@5,recall@10,recall@20,recall@30,recall@50,recall@100,OFFICIAL_recall,OFFICIAL_precision
0,2026-09-20T15:15:50,v3b_ft_reranker_improved,bm25,chunk=3000 k1=0.9 b=0.4,0.69175,0.79025,0.859667,0.893667,0.917917,0.948417,0.69175,0.1486


---
## Stage 5 — Dense retrieval

Bước nặng duy nhất. **Corpus là cố định và dùng chung cho cả train lẫn private test** →
encode **một lần**, tái dùng cho mọi thí nghiệm sau. Đây là điểm tiết kiệm lớn nhất trên Colab.

Encode theo **shard 50k chunk, ghi từng shard ra Drive ngay**. Session Colab chết ở phút
thứ 50 mà không có shard là mất trắng cả tiếng GPU; có shard thì chỉ chạy lại phần thiếu.

Vector lưu **fp16** (~500k × 768 × 2B ≈ 750MB) — vừa RAM, vừa VRAM T4, không cần faiss:
`q @ emb.T` trên GPU đã đủ nhanh và khỏi phải build index.

In [11]:
#@title Encode corpus (đa GPU, sharded, resume được)
import numpy as np, os, glob, time, re, torch
from sentence_transformers import SentenceTransformer

def set_dense_model(model_path):
    """Chọn checkpoint cho nhánh dense + trỏ EMB_DIR sang cache của đúng checkpoint đó.

    Tách thành hàm vì Stage 8 gọi lại nó sau khi fine-tune: model đổi thì embedding
    cũ vô giá trị, phải đổi cả nơi ghi lẫn nơi đọc chứ không chỉ đổi mỗi model.
    """
    global MODEL_PATH, TAG, EMB_REL, EMB_DIR, EMB_DIRS
    MODEL_PATH = model_path
    _is_ft = os.path.basename(os.path.normpath(MODEL_PATH)) == f'biencoder-ft-{PROFILE}'
    _model_key = f'biencoder_ft_{PROFILE}' if _is_ft else re.sub(r'[^a-zA-Z0-9]+', '_', MODEL_PATH)
    TAG     = _model_key + f'_c{CHUNK_CHARS}{_CAPTAG}'
    EMB_REL = f'emb/{TAG}'
    EMB_DIR = f'{ART}/{EMB_REL}'          # nơi GHI
    os.makedirs(EMB_DIR, exist_ok=True)
    # nơi ĐỌC: thư mục ghi + mọi cache chỉ đọc (Kaggle: output session trước attach lại)
    EMB_DIRS = [EMB_DIR] + [f'{d}/{EMB_REL}' for d in ART_IN if os.path.isdir(f'{d}/{EMB_REL}')]
    print('model    ->', MODEL_PATH)
    print('ghi      ->', EMB_DIR)
    if len(EMB_DIRS) > 1:
        print('đọc thêm <-', EMB_DIRS[1:])

def _shard_file(a):
    """shard đã encode ở BẤT KỲ cache nào -> path, chưa có -> None."""
    for d in EMB_DIRS:
        p = f'{d}/s_{a:08d}.npy'
        if os.path.exists(p):
            return p
    return None

def _l2(v):
    """normalize tay: encode_multi_process ở một số bản ST không nhận
    normalize_embeddings, làm ở đây thì mọi bản đều cho ra vector cùng thang."""
    v = np.asarray(v, dtype=np.float32)
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v

def encode_corpus():
    texts  = format_dense_passages(chunks.text.tolist())
    shards = [(i, min(i + ENC_SHARD, len(texts))) for i in range(0, len(texts), ENC_SHARD)]
    todo   = [(a, b) for a, b in shards if _shard_file(a) is None]

    if todo:
        model = SentenceTransformer(MODEL_PATH, device=DEV)
        model.max_seq_length = MAX_SEQ
        if torch.cuda.is_available():
            model.half()

        # >1 GPU: mỗi GPU một process con, ST tự chia đều sentence cho các process.
        # Đây là chỗ ăn tốc độ lớn nhất của Kaggle 2×GPU — encode là bước nặng nhất.
        pool = None
        if len(DEVICES) > 1:
            try:
                pool = model.start_multi_process_pool(target_devices=DEVICES)
                print(f'multi-GPU pool: {DEVICES}', flush=True)
            except Exception as e:
                print('không mở được multi-process pool (%s) -> chạy 1 GPU' % e, flush=True)
                pool = None

        print(f'{len(todo)}/{len(shards)} shard cần encode', flush=True)
        try:
            for a, b in todo:
                t0 = time.time()
                if pool is not None:
                    v = model.encode_multi_process(
                        texts[a:b], pool, batch_size=ENC_BATCH,
                        chunk_size=max(500, (b - a) // (4 * len(DEVICES))))
                else:
                    v = model.encode(texts[a:b], batch_size=ENC_BATCH,
                                     convert_to_numpy=True, show_progress_bar=False)
                np.save(f'{EMB_DIR}/s_{a:08d}.npy', _l2(v).astype(np.float16))
                print(f'  shard {a}-{b}: {time.time()-t0:.0f}s', flush=True)
        finally:
            if pool is not None:
                model.stop_multi_process_pool(pool)
            del model
            torch.cuda.empty_cache()

    return np.concatenate([np.load(_shard_file(a)) for a, _ in shards])

set_dense_model(FT_BIENCODER_PATH if USE_FINETUNED else BIENCODER)
t0  = time.time()
EMB = encode_corpus()
assert len(EMB) == len(chunks), f'emb {len(EMB)} != chunks {len(chunks)} — xoá {EMB_DIR} và encode lại'
print('EMB', EMB.shape, EMB.dtype, '| %.1f MB | %.1fs' % (EMB.nbytes / 1e6, time.time() - t0))


model    -> /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/models/biencoder-ft-v3b_ft_reranker_improved
ghi      -> /kaggle/working/DSC2026_Task1/artifacts/emb/biencoder_ft_v3b_ft_reranker_improved_c3000_m2500_t1
đọc thêm <- ['/kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/emb/biencoder_ft_v3b_ft_reranker_improved_c3000_m2500_t1']
EMB (151034, 1024) float16 | 309.3 MB | 5.3s


In [12]:
#@title Dense search trên GPU
import torch, numpy as np
from sentence_transformers import SentenceTransformer

# EMB fp16 chỉ vài trăm MB nên để trọn trên 1 GPU: q @ EMB.T là một phép matmul,
# chia cho 2 GPU chỉ thêm đồng bộ chứ không nhanh hơn đáng kể. GPU còn lại được
# dùng thật sự ở Stage 5 (encode) và Stage 7 (rerank) — hai bước nặng.
EMB_T  = None
qmodel = None

def free_dense():
    """Nhả EMB_T + query encoder khỏi VRAM. Stage 8 gọi trước khi train: chỗ này
    đang giữ vài GB, để nguyên thì fine-tune gần như chắc chắn OOM."""
    global EMB_T, qmodel
    EMB_T = qmodel = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_query_model():
    """Đưa EMB lên GPU + nạp encoder cho query theo MODEL_PATH hiện tại.
    Gọi lại được — Stage 8 dùng để chuyển sang model vừa fine-tune."""
    global EMB_T, qmodel
    free_dense()
    EMB_T = torch.from_numpy(EMB).to(DEV)                   # (N, dim) fp16
    qmodel = SentenceTransformer(MODEL_PATH, device=DEV)
    qmodel.max_seq_length = MAX_SEQ
    if DEV.startswith('cuda'):
        qmodel.half()

def dense_search(qtext, topk=CAND_CHUNKS, batch=32):
    """-> (indices, scores) của top chunk cho mỗi query."""
    ixs, scs = [], []
    for i in range(0, len(qtext), batch):
        q = qmodel.encode(format_dense_queries(qtext[i:i+batch]), batch_size=batch, normalize_embeddings=True,
                          convert_to_tensor=True, show_progress_bar=False).to(EMB_T.dtype)
        s = q @ EMB_T.T                                     # cosine (đã normalize)
        v, ix = torch.topk(s, min(topk, s.shape[1]), dim=1)
        ixs.append(ix.cpu().numpy()); scs.append(v.float().cpu().numpy())
    return np.concatenate(ixs), np.concatenate(scs)

def retrieve_dense(qids, qtext):
    ix, sc = dense_search(qtext)
    return {q: chunks_to_docs(s, i)[0] for q, i, s in zip(qids, ix, sc)}

load_query_model()
t0 = time.time()
val_text = [QUESTIONS[q] for q in VAL_QIDS]
dense_val = retrieve_dense(VAL_QIDS, val_text)
print('%.1fs' % (time.time() - t0))
m = show('dense' + ('-ft' if USE_FINETUNED else ''), dense_val)
log_exp('dense_ft' if USE_FINETUNED else 'dense', m, MODEL_PATH)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2.9s
dense-ft                     recall@5=0.9283 recall@10=0.9606 recall@20=0.9794 recall@30=0.9827 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9283 OFFICIAL_precision=0.2002


,time,profile,run,note,recall@5,recall@10,recall@20,recall@30,recall@50,recall@100,OFFICIAL_recall,OFFICIAL_precision
0,2026-09-20T15:15:50,v3b_ft_reranker_improved,bm25,chunk=3000 k1=0.9 b=0.4,0.691750,0.790250,0.859667,0.893667,0.917917,0.948417,0.691750,0.1486
1,2026-09-20T15:16:22,v3b_ft_reranker_improved,dense_ft,/kaggle/input/datasets/kitnehi1211/v3b-finetun...,0.928333,0.960583,0.979417,0.982667,0.984917,0.987917,0.928333,0.2002


---
## Stage 6 — Hybrid fusion (RRF **có trọng số**)

Điểm BM25 và điểm cosine nằm trên hai thang hoàn toàn khác nhau nên chuẩn hoá chúng
luôn mong manh — RRF chỉ dùng **hạng**, bền hơn nhiều:

`score(d) = Σ wᵢ / (K + rankᵢ(d))`

### Vì sao có `wᵢ` — bài học từ profile v2

| | recall@5 |
|---|---|
| BM25 | 0.6868 |
| dense | **0.8996** |
| hybrid RRF w=(1,1) | 0.8699 |

RRF cổ điển cho mọi kênh **một phiếu bằng nhau**. BM25 yếu hơn dense 21 điểm mà vẫn
được bỏ phiếu ngang cơ → fusion tụt 3 điểm so với chỉ dùng dense. Đây không phải lỗi
cài đặt, đó là RRF đang làm đúng thứ nó được bảo làm.

Cell dưới sweep `w = (1,1) … (1,5)` **và** cả phương án bỏ hẳn BM25, rồi chọn theo val.
Sweep này thuần CPU, không đụng tới quota GPU — không có lý do gì để đoán.

> BM25 vẫn được giữ lại vì nó phục vụ hai việc khác: chọn chunk đại diện cho doc ở
> Stage 7 (doc lọt vào ứng viên nhờ từ khoá hiếm thường không có chunk nào trong top
> dense) và mine hard negative ở Stage 8.


In [13]:
#@title RRF fuse (có trọng số) + sweep chọn trọng số theo val
def rrf(*rankings, k=RRF_K, topn=CAND_DOCS, weights=None):
    """Mỗi ranking: {qid: [doc_id]} -> {qid: [doc_id] đã fuse}.

    `weights` là điểm mới so với bản trước. RRF cổ điển cho mọi kênh một phiếu
    bằng nhau, và đó chính là lý do hybrid v2 (0.8699) THUA dense đơn thuần
    (0.8996): BM25 chỉ đạt recall@5 = 0.6868 nhưng vẫn được quyền bỏ phiếu ngang
    dense. Kênh yếu hơn phải có phiếu nhẹ hơn.
    """
    ws = list(weights) if weights else [1.0] * len(rankings)
    assert len(ws) == len(rankings)
    out = {}
    for q in rankings[0]:
        s = {}
        for w, r in zip(ws, rankings):
            for rank, d in enumerate(r.get(q, [])):
                s[d] = s.get(d, 0.0) + w / (k + rank + 1)
        out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])[:topn]]
    return out


# Sweep trọng số: rẻ (thuần CPU, không đụng GPU) nên đo hết rồi chọn, đừng đoán.
# dense đơn thuần cũng là một ứng viên hợp lệ — và theo log v2 nó đang dẫn đầu.
_cands = {'dense (bỏ hẳn BM25)': dense_val}
for _w in [(1, 1), (1, 2), (1, 3), (1, 5)]:
    _cands[f'hybrid RRF w={_w}'] = rrf(bm25_val, dense_val, weights=_w)

_scores = {}
for _n, _p in _cands.items():
    _scores[_n] = show(_n, _p)

_best_name = max(_scores, key=lambda n: (_scores[n]['OFFICIAL_recall'],
                                         _scores[n]['OFFICIAL_precision']))
print(f'\n-> val chọn: {_best_name}  (recall@5 = {_scores[_best_name]["recall@5"]:.4f})')

# hybrid_val = nhánh RRF tốt nhất (KHÔNG phải dense — Stage 7 so hai base riêng).
_best_w = max([w for w in [(1, 1), (1, 2), (1, 3), (1, 5)]],
              key=lambda w: _scores[f'hybrid RRF w={w}']['OFFICIAL_recall'])
if tuple(RRF_WEIGHTS) != tuple(_best_w):
    print(f'   RRF_WEIGHTS ở Stage 0 đang là {tuple(RRF_WEIGHTS)}, val thích {_best_w}'
          ' -> sửa Stage 0 cho khớp trước khi chạy Stage 9.')
hybrid_val = _cands[f'hybrid RRF w={_best_w}']
log_exp('hybrid_rrf_w', _scores[f'hybrid RRF w={_best_w}'], f'K={RRF_K} w={_best_w}')


dense (bỏ hẳn BM25)          recall@5=0.9283 recall@10=0.9606 recall@20=0.9794 recall@30=0.9827 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9283 OFFICIAL_precision=0.2002
hybrid RRF w=(1, 1)          recall@5=0.8863 recall@10=0.9443 recall@20=0.9691 recall@30=0.9792 recall@50=0.9877 recall@100=0.9894 OFFICIAL_recall=0.8863 OFFICIAL_precision=0.1900
hybrid RRF w=(1, 2)          recall@5=0.9137 recall@10=0.9618 recall@20=0.9774 recall@30=0.9837 recall@50=0.9862 recall@100=0.9904 OFFICIAL_recall=0.9137 OFFICIAL_precision=0.1966
hybrid RRF w=(1, 3)          recall@5=0.9260 recall@10=0.9648 recall@20=0.9809 recall@30=0.9837 recall@50=0.9864 recall@100=0.9879 OFFICIAL_recall=0.9260 OFFICIAL_precision=0.1996
hybrid RRF w=(1, 5)          recall@5=0.9307 recall@10=0.9673 recall@20=0.9807 recall@30=0.9847 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9307 OFFICIAL_precision=0.2008

-> val chọn: hybrid RRF w=(1, 5)  (recall@5 = 0.9307)
   RRF_WEIGHTS ở Stage 0 đang là (1.0, 3.0), 

,time,profile,run,note,recall@5,recall@10,recall@20,recall@30,recall@50,recall@100,OFFICIAL_recall,OFFICIAL_precision
0,2026-09-20T15:15:50,v3b_ft_reranker_improved,bm25,chunk=3000 k1=0.9 b=0.4,0.691750,0.790250,0.859667,0.893667,0.917917,0.948417,0.691750,0.1486
1,2026-09-20T15:16:22,v3b_ft_reranker_improved,dense_ft,/kaggle/input/datasets/kitnehi1211/v3b-finetun...,0.928333,0.960583,0.979417,0.982667,0.984917,0.987917,0.928333,0.2002
2,2026-09-20T15:16:22,v3b_ft_reranker_improved,hybrid_rrf_w,"K=60 w=(1, 5)",0.930750,0.967333,0.980667,0.984667,0.984917,0.987917,0.930750,0.2008


---
## Stage 7 — Cross-encoder rerank

Cross-encoder đọc `(question, chunk)` cùng lúc nên hiểu quan hệ sâu hơn bi-encoder,
nhưng đắt gấp hàng nghìn lần → chỉ chấm `RERANK_TOP` doc.

### Đây là khâu duy nhất còn dư địa — và nó đang chưa làm được việc

Profile v2, val 1.000 câu, corpus đầy đủ 8.532 doc:

| | recall@5 | recall@100 |
|---|---|---|
| dense | **0.8996** | 0.9897 |
| hybrid RRF w=(1,1) | 0.8699 | 0.9905 |
| hybrid + rerank replace | 0.8768 | — |
| hybrid + rerank RRF | 0.8973 | — |

Đọc bảng này ra hai kết luận:

1. **Retriever đã xong việc.** `recall@100 = 0.99` — 99% số câu có gold nằm trong
   danh sách ứng viên. Thêm kênh retrieval mới (char n-gram, số hiệu văn bản,
   query expansion) là tối ưu một thứ đã hết chỗ tối ưu.
2. **Không cấu hình nào vượt nổi dense đơn thuần.** Cả BM25 fusion lẫn cross-encoder
   đều đang âm. Trần của rerank trên top-30 là `recall@30 ≈ 0.975`, đang đạt 0.90 →
   **~7,5 điểm nằm trên bàn, thuần tuý từ xếp hạng.**

### Ba thay đổi ở Stage này

- **`RERANK_CHUNKS_PER_DOC = 3`** thay vì 1. Với nhiều chunk trên mỗi document và
  `MAX_CHUNKS_PER_DOC=150`, reranker trước đây chỉ được đọc **1/150** nội dung mỗi
  văn bản. Chọn nhầm chunk đại diện là gold doc bị đạp xuống dù nội dung đúng nằm ở
  Điều khác. Giờ chấm 3 chunk rồi max-pool.
- **So `base='dense'` với `base='hybrid'`.** v2 rerank trên `hybrid_val` — tức là đưa
  cho cross-encoder một danh sách đã bị BM25 làm hỏng. Cell dưới chấm cả hai.
- **Cache `_RR_SCORE[(qid, doc_id)]`.** Top-30 của hai base trùng nhau phần lớn, nên
  base thứ hai gần như không tốn thêm GPU.

### Chẩn đoán kèm theo

Nếu reranker chấm một danh sách `recall@30 ≈ 0.96` mà chỉ
ra `recall@5 = 0.877` là **bất thường**, không chỉ là "yếu". Cell dưới in phân bố điểm
để loại trừ hai thủ phạm hay gặp: fp16 tràn số (NaN/Inf) và model load sai (điểm gần
như hằng số → xếp hạng thành ngẫu nhiên). Nếu phân bố bình thường mà điểm vẫn thấp
thì vấn đề là **domain**, và việc tiếp theo là fine-tune cross-encoder — không phải
thêm kênh retrieval.


In [14]:
#@title Chọn N chunk đại diện cho mỗi doc ứng viên
import numpy as np
from collections import defaultdict

def bm25_topk(qtext, batch=64):
    """bm25.search() dựng ma trận dense (nq × n_chunk) -> phải chấm theo lô,
    1000 query × ~500k chunk một lần là ~2GB RAM."""
    ixs, scs = [], []
    for i in range(0, len(qtext), batch):
        ix, sc = bm25.search(qtext[i:i+batch])
        ixs.append(ix); scs.append(sc)
    return np.concatenate(ixs), np.concatenate(scs)


def best_chunks_per_doc(qids, qtext, cand, m=None):
    """{qid: {doc_id: [chunk_ix, ...]}} — tối đa `m` chunk cho mỗi doc.

    Bản trước trả về ĐÚNG MỘT chunk/doc. Với nhiều chunk trên mỗi document và
    MAX_CHUNKS_PER_DOC=150, cross-encoder khi đó chỉ được đọc 1/150 nội dung
    văn bản — chọn nhầm chunk là gold doc bị đạp xuống dù nội dung đúng nằm ở
    Điều khác. Lấy m chunk rồi max-pool bịt đúng lỗ đó.

    Nguồn chunk, theo thứ tự ưu tiên (không trộn hai thang điểm với nhau):
        1. chunk có điểm dense cao nhất trong doc
        2. bù thêm bằng chunk BM25 cao nhất — quan trọng với doc lọt vào ứng
           viên nhờ BM25, loại này thường không có chunk nào trong top dense
        3. vẫn thiếu (hiếm) -> chunk đầu văn bản (chứa số hiệu, trích yếu)
    """
    m = m or RERANK_CHUNKS_PER_DOC
    d_ix, d_sc = dense_search(qtext, topk=CAND_CHUNKS)
    b_ix, b_sc = bm25_topk(qtext)

    best = {}
    for n, q in enumerate(qids):
        want = set(cand[q])
        dense_by_doc, bm25_by_doc = defaultdict(list), defaultdict(list)
        for ci, s in zip(d_ix[n], d_sc[n]):
            d = DOC_IDS[CHUNK2DOC[ci]]
            if d in want:
                dense_by_doc[d].append((s, ci))
        for ci, s in zip(b_ix[n], b_sc[n]):
            d = DOC_IDS[CHUNK2DOC[ci]]
            if d in want:
                bm25_by_doc[d].append((s, ci))

        out = {}
        for d in want:
            picked, seen = [], set()
            for _, ci in sorted(dense_by_doc.get(d, []), key=lambda x: -x[0]):
                if ci not in seen:
                    seen.add(ci); picked.append(int(ci))
                if len(picked) >= m:
                    break
            if len(picked) < m:
                for _, ci in sorted(bm25_by_doc.get(d, []), key=lambda x: -x[0]):
                    if ci not in seen:
                        seen.add(ci); picked.append(int(ci))
                    if len(picked) >= m:
                        break
            if not picked:
                picked = [int(np.searchsorted(CHUNK2DOC, DOC2IX[d]))]
            out[d] = picked
        best[q] = out
    return best


def best_chunk_per_doc(qids, qtext, cand):
    """API cũ: đúng 1 chunk/doc. Stage 8 (mine hard negative) vẫn dùng bản này —
    triplet cho MultipleNegativesRankingLoss cần một đoạn văn bản, không phải list."""
    return {q: {d: cis[0] for d, cis in v.items()}
            for q, v in best_chunks_per_doc(qids, qtext, cand, m=1).items()}


CHUNK_TEXTS = chunks.text.tolist()
print('ok')


ok


In [15]:
#@title Rerank (N chunk/doc, cache điểm, chia query cho từng GPU)
from sentence_transformers import CrossEncoder
import numpy as np, time, torch, threading

RR_DEVICES = DEVICES                       # 1 reranker / GPU
RERANKERS  = {}

# Cache điểm cross-encoder theo (qid, doc_id). Nhờ nó, rerank base thứ hai
# (hybrid sau khi đã chạy dense, hoặc ngược lại) chỉ tốn GPU cho phần cặp MỚI —
# hai base top-30 trùng nhau phần lớn. Reranker đổi model thì phải .clear().
_RR_SCORE = {}

def get_reranker(dev):
    if dev not in RERANKERS:
        m = CrossEncoder(RERANKER, max_length=RERANK_MAXLEN, device=dev,
                         activation_fn=torch.nn.Identity())
        if RERANK_FP16 and dev.startswith('cuda'):
            m.model.half()
        RERANKERS[dev] = m
    return RERANKERS[dev]

# Nạp trước, tuần tự: hai thread cùng tải model một lúc sẽ tranh cache HuggingFace.
if False:  # production: lazy-load ở Stage 8B.3
    for _d in RR_DEVICES:
        get_reranker(_d)
        print('reranker ->', _d, flush=True)
else:
    print('RUN_PHASE=1: hoãn load reranker tới Stage 8B.2')


def rerank(qids, qtext, cand, top=RERANK_TOP, m=None):
    """Xếp lại top-`top` doc bằng cross-encoder; điểm doc = MAX trên `m` chunk.

    Mỗi GPU nhận một phần danh sách query và chạy độc lập trong một thread.
    Forward của HF nhả GIL khi chờ CUDA nên 2 thread thực sự chạy song song.
    """
    m   = m or RERANK_CHUNKS_PER_DOC
    txt = dict(zip(qids, qtext))
    sel = {q: cand[q][:top] for q in qids}

    need = {q: [d for d in sel[q] if (q, d) not in _RR_SCORE] for q in qids}
    todo = [q for q in qids if need[q]]
    n_pair = sum(len(need[q]) for q in todo) * m
    print(f'  {len(todo)}/{len(qids)} query cần chấm | ~{n_pair} cặp '
          f'({m} chunk/doc)', flush=True)

    if todo:
        rep   = best_chunks_per_doc(todo, [txt[q] for q in todo], need, m)
        jobs  = [(q, txt[q], need[q], rep[q]) for q in todo]
        parts = [jobs[i::len(RR_DEVICES)] for i in range(len(RR_DEVICES))]
        lock, done = threading.Lock(), [0]

        def work(dev, part):
            mdl = get_reranker(dev)
            for q, qt, docs, chunk_of in part:
                pairs, owner = [], []
                for d in docs:
                    for ci in chunk_of[d]:
                        pairs.append([qt, CHUNK_TEXTS[ci]]); owner.append(d)
                if pairs:
                    s = mdl.predict(pairs, batch_size=RERANK_BATCH,
                                    show_progress_bar=False)
                    # điểm doc = MAX trên các chunk: gold là "văn bản CHỨA thông
                    # tin cần thiết" -> một Điều khớp là đủ, mean sẽ phạt oan
                    # văn bản dài.
                    pooled = {}
                    for d, v in zip(owner, np.asarray(s, dtype=np.float64)):
                        if v > pooled.get(d, -1e30):
                            pooled[d] = float(v)
                    with lock:
                        _RR_SCORE.update({(q, d): v for d, v in pooled.items()})
                with lock:
                    done[0] += 1
                    if done[0] % 100 == 0:
                        print(f'  {done[0]}/{len(jobs)}', flush=True)

        ths = [threading.Thread(target=work, args=(d, p), daemon=True)
               for d, p in zip(RR_DEVICES, parts) if p]
        for t in ths: t.start()
        for t in ths: t.join()

    # sorted() ổn định -> doc bằng điểm giữ nguyên thứ tự của retriever.
    # Doc ngoài top rerank vẫn nối phía sau, không mất khỏi danh sách.
    return {q: sorted(sel[q], key=lambda d: -_RR_SCORE.get((q, d), -1e30)) + cand[q][top:]
            for q in qids}


def fuse_rerank(cand, reranked, k=RRF_K, weights=(1.0, 1.0)):
    """RRF giữa thứ tự retriever và thứ tự sau rerank.

    'replace' đặt toàn bộ niềm tin vào cross-encoder: nó xếp nhầm một câu là câu
    đó mất trắng, dù retriever đã để gold ở hạng 1. RRF giữ lại phiếu của
    retriever nên hỏng ít hơn — đổi lại phần thắng cũng nhỏ hơn khi reranker đúng.
    """
    wc, wr = weights
    out = {}
    for q, docs in reranked.items():
        s = {}
        for r, d in enumerate(cand[q]):
            s[d] = s.get(d, 0.0) + wc / (k + r + 1)
        for r, d in enumerate(docs):
            s[d] = s.get(d, 0.0) + wr / (k + r + 1)
        out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])]
    return out


def apply_rerank(qids, qtext, cand, mode=None):
    """Một cửa duy nhất cho cả val lẫn public -> hai bên không thể lệch nhau."""
    mode = mode or RERANK_MODE
    if mode == 'none':
        return cand
    r = rerank(qids, qtext, cand)
    return r if mode == 'replace' else fuse_rerank(cand, r)


BASES = {'dense': dense_val, 'hybrid': hybrid_val}

if False:  # production: bỏ benchmark zero-shot cũ
    # ---- So base dense vs hybrid trong CÙNG một lần chạy -------------------------
    # Log v2 rerank trên `hybrid_val` (0.8699) — tức là cross-encoder được đưa một
    # danh sách đã bị BM25 làm hỏng. Ở đây chấm cả hai base; nhờ _RR_SCORE, base thứ
    # hai gần như không tốn thêm GPU.

    t0, results = time.time(), {}
    for _b, _cand in BASES.items():
        print(f'[base={_b}]', flush=True)
        _rr = rerank(VAL_QIDS, val_text, _cand)
        results[(_b, 'none')]    = show(f'[{_b}] không rerank', _cand)
        results[(_b, 'replace')] = show(f'[{_b}] rerank replace', _rr)
        results[(_b, 'rrf')]     = show(f'[{_b}] rerank fuse RRF', fuse_rerank(_cand, _rr))
        if _b == RERANK_BASE:
            rerank_val, rerank_rrf_val = _rr, fuse_rerank(_cand, _rr)
    print('%.1f phút (%d GPU)' % ((time.time() - t0) / 60, len(RR_DEVICES)))

    # ---- Chẩn đoán: reranker có thật sự chạy đúng không -------------------------
    # Nếu reranker chấm một danh sách recall@30 ~0.96 mà chỉ
    # ra recall@5 = 0.877 là BẤT THƯỜNG. Hai thủ phạm hay gặp, cả hai lộ ra ở đây:
    #   - fp16 tràn số  -> NaN/Inf, hoặc điểm dồn cục về vài giá trị
    #   - model load sai -> điểm gần như hằng số, xếp hạng thành ngẫu nhiên
    _v = np.array(list(_RR_SCORE.values()), dtype=np.float64)
    print('\nđiểm reranker: n=%d  min=%.4f  p50=%.4f  max=%.4f  std=%.4f'
          % (len(_v), _v.min(), np.median(_v), _v.max(), _v.std()))
    print('giá trị phân biệt: %.1f%%  (thấp -> reranker không phân biệt được ứng viên)'
          % (100.0 * len(np.unique(np.round(_v, 4))) / max(1, len(_v))))
    if not np.isfinite(_v).all():
        print('  !! có NaN/Inf -> đặt RERANK_FP16 = False ở Stage 0 rồi chạy lại cell này')
    if _v.std() < 1e-3:
        print('  !! điểm gần như hằng số -> reranker đang KHÔNG hoạt động, kiểm tra RERANKER/prompt')

    for _k, _m in results.items():
        log_exp(f'rerank_{_k[0]}_{_k[1]}', _m,
                f'{RERANKER} top{RERANK_TOP} x{RERANK_CHUNKS_PER_DOC}chunk fp16={RERANK_FP16}')

    _best = max(results, key=lambda k: (results[k]['OFFICIAL_recall'],
                                        results[k]['OFFICIAL_precision']))
    print(f"\n-> val chọn RERANK_BASE = '{_best[0]}', RERANK_MODE = '{_best[1]}'"
          f"  (recall@5 = {results[_best]['recall@5']:.4f})")
    print(f"   đang đặt ở Stage 0: RERANK_BASE = '{RERANK_BASE}', RERANK_MODE = '{RERANK_MODE}'")
    if (RERANK_BASE, RERANK_MODE) != _best:
        print('   -> sửa hai biến đó ở Stage 0 rồi chạy lại Stage 9 (KHÔNG cần encode lại).')

    # Trần của Stage 7: doc không lọt top-RERANK_TOP thì reranker không cứu được.
    # recall@RERANK_TOP có trong metrics khi RERANK_TOP thuộc ks của evaluate();
    # đổi RERANK_TOP sang giá trị lạ thì lùi về @50 rồi @100 làm cận trên.
    _base_m = results[(_best[0], 'none')]
    _ck = next((k for k in (f'recall@{RERANK_TOP}', 'recall@50', 'recall@100')
                if k in _base_m), 'recall@100')
    _ceil = _base_m[_ck]
    print(f"\ntrần của rerank (recall@{RERANK_TOP} của base '{_best[0]}') = {_ceil:.4f}"
          f"  |  đang đạt {results[_best]['recall@5']:.4f}"
          f"  |  còn {100*(_ceil - results[_best]['recall@5']):.1f} điểm trong tầm với")
    print('nếu khoảng cách này còn lớn -> fine-tune cross-encoder là việc tiếp theo,'
          ' không phải thêm kênh retrieval.')
else:
    print('RUN_PHASE=1: bỏ benchmark reranker zero-shot; train dùng dense candidates trực tiếp.')


RUN_PHASE=1: hoãn load reranker tới Stage 8B.2
RUN_PHASE=1: bỏ benchmark reranker zero-shot; train dùng dense candidates trực tiếp.


---
## Stage 7B — Sửa lỗi chọn chunk + đo lại cho tử tế

Khối này **không sửa cell nào phía trên**: mọi số liệu cũ vẫn còn để đối chiếu.

### Lỗi đang làm reranker mất tác dụng

`best_chunks_per_doc` (Stage 7) chỉ xét chunk nằm trong **top-`CAND_CHUNKS`=1000 của
TOÀN corpus**. Đo trên dữ liệu thật:

| Đo trên `train.json` + corpus | Giá trị |
|---|---|
| Tổng chunk (`CHUNK_CHARS=3000`, cap 150) | **~154.000** (markdown Stage 4/5 ghi ~500k là số cũ) |
| Chunk/doc, toàn corpus | mean 18 · p50 11 |
| Chunk/doc **của gold doc** | mean 45 · **p50 30** · p90 117 |
| Gold doc bị `MAX_CHUNKS_PER_DOC=150` cắt | **6,3%** số lần xuất hiện (chỉ 41 doc bị cap) |
| Độ dài gold vs corpus | p50 64k vs 22k ký tự |

Một doc xếp hạng ~25 thường chỉ có **1–2 chunk** lọt top-1000 toàn cục → yêu cầu
`RERANK_CHUNKS_PER_DOC=3` **không bao giờ được đáp ứng**. Tệ hơn: đúng những gold doc bị
retriever xếp thấp — chính là các ca reranker phải cứu — lại được cấp ít chunk nhất và
chọn kém nhất. Reranker đang đọc ~3/30 chunk của gold doc, tức **~10% nội dung**.

`select_chunks` ở dưới lấy top-m chunk **trong phạm vi từng doc** từ full similarity row.
`dense_search` đã tính row đó rồi, nên gần như miễn phí.

### Đo bằng paired bootstrap, không so hai giá trị mean

`N_VAL=1000`, recall@5 ≈ 0,9 → sai số chuẩn ≈ **0,0095**. Các con số trong log v2
(0,8768 / 0,8973 / 0,8996) lệch nhau đúng cỡ nhiễu đó. So sánh **ghép cặp theo từng câu**
phân biệt được hai config lệch mean 0,003 nếu chúng đồng thuận trên 95% câu.

### Grid: model yếu, hay input sai?

| tag | cặp/query | token/query | giả thuyết |
|---|---|---|---|
| `top20_m8_L512` | 160 | 82k | passage ngắn tập trung — **rẻ hơn** config hiện tại |
| `top30_m8_L768` | 240 | 184k | ngang chi phí, coverage 8 chunk/doc |
| `top30_m3_L2048` | 90 | 184k | = config hiện tại, nhưng selector đã sửa |

7B.3 lưu điểm **từng chunk** chứ không pool sẵn → toàn bộ sweep pooling × fusion ở 7B.4
chạy trên CPU, không tốn thêm một giây GPU.


In [16]:
#@title 7B.1 — So sánh có ý nghĩa thống kê (paired bootstrap)
# N_VAL=1000, recall@5 ~0.9 -> sai số chuẩn ~0.0095. Log v2 chọn config dựa trên
# chênh lệch 0.003-0.02, tức là nằm trong vùng nhiễu nếu chỉ so hai giá trị mean.
# Paired bootstrap so TỪNG CÂU trên cùng tập câu hỏi: hai config có thể lệch mean
# chỉ 0.003 mà vẫn phân biệt được, nếu chúng đồng thuận trên 95% câu.
import numpy as np

def per_query_recall(preds, gold, k=TOPK_SUBMIT):
    """-> (danh sách qid, vector recall theo từng câu). Nguyên liệu để ghép cặp."""
    qs = list(gold)
    return qs, np.array([len(gold[q] & set(preds.get(q, [])[:k])) / len(gold[q])
                         for q in qs])

def paired_bootstrap(a, b, n_boot=5000, seed=SEED):
    """a, b: vector recall/câu của hai config trên CÙNG tập câu, CÙNG thứ tự."""
    d = np.asarray(a, float) - np.asarray(b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boot = d[idx].mean(axis=1)
    return {'delta': float(d.mean()),
            'ci_lo': float(np.percentile(boot, 2.5)),
            'ci_hi': float(np.percentile(boot, 97.5)),
            'p_better': float((boot > 0).mean()),
            'win': int((d > 0).sum()), 'loss': int((d < 0).sum()),
            'tie': int((d == 0).sum())}

def compare(name_a, preds_a, name_b, preds_b, gold=None, k=TOPK_SUBMIT):
    """In delta + CI95 + P(A>B). Chỉ tin khi P(A>B) > 0.95 hoặc < 0.05."""
    gold = gold or {q: GOLD[q] for q in VAL_QIDS}
    _, a = per_query_recall(preds_a, gold, k)
    _, b = per_query_recall(preds_b, gold, k)
    r = paired_bootstrap(a, b)
    verdict = ('THẮNG rõ' if r['p_better'] > 0.95 else
               'THUA rõ'  if r['p_better'] < 0.05 else 'KHÔNG phân biệt được')
    print(f"  {name_a}  vs  {name_b}")
    print(f"    delta={r['delta']:+.4f}  CI95=[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  "
          f"P(A>B)={r['p_better']:.3f}")
    print(f"    thắng {r['win']} / thua {r['loss']} / bằng {r['tie']} câu  ->  {verdict}")
    return r

print('ok — dùng: compare("A", preds_a, "B", preds_b)')


ok — dùng: compare("A", preds_a, "B", preds_b)


In [17]:
#@title 7B.2 — select_chunks: top-m chunk TRONG từng doc (thay best_chunks_per_doc)
# Lỗi của bản cũ: nó chỉ xét chunk nằm trong top-CAND_CHUNKS=1000 của TOÀN corpus.
# Một doc xếp hạng ~25 thường chỉ có 1-2 chunk lọt top-1000 -> m=3 không bao giờ
# được đáp ứng, và đúng những gold doc bị retriever xếp thấp (ca mà reranker phải
# cứu) lại được cấp ít chunk nhất. Gold doc có p50 = 30 chunk -> reranker đang đọc
# ~10% nội dung.
#
# EMB chỉ ~154k chunk nên full similarity row (q @ EMB.T) đã được dense_search
# tính rồi; lấy top-m trong phạm vi chunk của từng doc là gần như miễn phí.
# CHUNK2DOC tăng dần (chunk sinh theo thứ tự corpus) -> searchsorted cho ngay
# khoảng [lo, hi) chunk của mỗi doc.
import numpy as np, torch

assert np.all(np.diff(CHUNK2DOC) >= 0), \
    'CHUNK2DOC không tăng dần -> searchsorted sai. Chunk phải sinh theo thứ tự corpus.'
_ar = np.arange(len(DOC_IDS))
DOC_CHUNK_LO = np.searchsorted(CHUNK2DOC, _ar, 'left')
DOC_CHUNK_HI = np.searchsorted(CHUNK2DOC, _ar, 'right')
_npd = DOC_CHUNK_HI - DOC_CHUNK_LO
assert _npd.sum() == len(chunks), 'map doc->chunk bị hổng'
print('chunk/doc: min %d  p50 %.0f  mean %.1f  max %d'
      % (_npd.min(), np.percentile(_npd, 50), _npd.mean(), _npd.max()))

# 20 văn bản trong corpus có `passage` RỖNG và cũng không có field `name` (tiêu đề
# chỉ được vá từ slug trong `link` ở Stage 1). chunk_doc() chỉ chunk passage nên
# chúng có 0 chunk -> dense retrieval KHÔNG BAO GIỜ truy hồi được. 6/20 doc này
# từng là gold và 9 câu hỏi train mất trắng vì mọi gold của chúng nằm trong nhóm
# đó. Đây là giới hạn của chunker ở Stage 2, không phải lỗi của cell này: bản B vá
# bằng chunk chỉ-tiêu-đề, còn bản A giữ nguyên chunk set để dùng tiếp cache cũ.
_empty = np.flatnonzero(_npd == 0)
if len(_empty):
    def _unreachable(q):
        gs = [g for g in GOLD[q] if g in DOC2IX]
        return bool(gs) and all(_npd[DOC2IX[g]] == 0 for g in gs)
    _lost = [q for q in GOLD if _unreachable(q)]
    _lost_val = [q for q in _lost if q in set(VAL_QIDS)]
    print(f'{len(_empty)} doc có 0 chunk (passage rỗng) -> không thể truy hồi. '
          f'Ví dụ: {[DOC_IDS[i] for i in _empty[:5]]}')
    print(f'  {len(_lost)} câu train có TOÀN BỘ gold trong nhóm đó '
          f'({len(_lost_val)} câu thuộc val) -> trần recall bị hạ đúng bằng phần này')

def select_chunks(qids, qtext, cand, m, batch=32):
    """-> {qid: {doc_id: [(chunk_ix, dense_score), ...]}}, giảm dần theo điểm.

    Trả kèm dense score để cell fusion dùng làm điểm doc (max chunk) mà không
    phải chạy lại retrieval lần nữa.
    """
    qids, out = list(qids), {}
    for i0 in range(0, len(qtext), batch):
        qv = qmodel.encode(format_dense_queries(qtext[i0:i0 + batch]),
                           batch_size=batch, normalize_embeddings=True,
                           convert_to_tensor=True, show_progress_bar=False).to(EMB_T.dtype)
        S = (qv @ EMB_T.T).float().cpu().numpy()          # (b, n_chunk)
        for r, q in enumerate(qids[i0:i0 + batch]):
            o = {}
            for d in cand[q]:
                lo, hi = DOC_CHUNK_LO[DOC2IX[d]], DOC_CHUNK_HI[DOC2IX[d]]
                if hi <= lo:            # doc passage rỗng -> không có chunk để chấm
                    continue
                loc = S[r, lo:hi]
                k = min(m, hi - lo)
                top = np.argpartition(-loc, k - 1)[:k]
                top = top[np.argsort(-loc[top])]
                o[d] = [(int(lo + j), float(loc[j])) for j in top]
            out[q] = o
    return out

# Kiểm chứng ngay: bản cũ cấp được bao nhiêu chunk/doc so với bản mới?
_probe = VAL_QIDS[:50]
_pt    = [QUESTIONS[q] for q in _probe]
_cand  = {q: BASES[RERANK_BASE][q][:RERANK_TOP] for q in _probe}
_old   = best_chunks_per_doc(_probe, _pt, _cand, m=RERANK_CHUNKS_PER_DOC)
_new   = select_chunks(_probe, _pt, _cand, m=RERANK_CHUNKS_PER_DOC)
_o = np.mean([len(v) for q in _probe for v in _old[q].values()])
_n = np.mean([len(v) for q in _probe for v in _new[q].values()])
print(f'chunk/doc thực nhận (m={RERANK_CHUNKS_PER_DOC}): '
      f'selector CŨ = {_o:.2f}  |  selector MỚI = {_n:.2f}')


chunk/doc: min 1  p50 10  mean 17.7  max 2354
chunk/doc thực nhận (m=8): selector CŨ = 6.55  |  selector MỚI = 7.57


In [18]:
#@title 7B.3 — rerank_collect: chấm và lưu điểm CHUNK-LEVEL
import numpy as np, time, threading
from collections import defaultdict

# Lưu điểm từng chunk, KHÔNG pool sẵn -> toàn bộ sweep pooling/fusion ở cell sau
# chạy trên CPU, không tốn thêm một giây GPU nào. Đây là lý do cell này đắt mà
# cell sau miễn phí.
RRC       = {}    # {tag: {qid: {doc_id: [điểm từng chunk]}}}
RRC_DENSE = {}    # {tag: {qid: {doc_id: điểm dense của chunk tốt nhất}}}
RRC_META  = {}    # {tag: (top, m, maxlen)}

def set_rr_maxlen(L):
    """Đổi max_length của mọi reranker đã nạp, không phải tải lại model."""
    for _m in RERANKERS.values():
        _m.max_length = L

def rerank_collect(qids, qtext, cand, top, m, maxlen, tag=None):
    qids = list(qids)
    tag  = tag or f'top{top}_m{m}_L{maxlen}'
    if tag in RRC:
        print(f'[{tag}] đã có trong RRC -> bỏ qua'); return tag
    set_rr_maxlen(maxlen)
    sel = {q: cand[q][:top] for q in qids}

    t0  = time.time()
    rep = select_chunks(qids, qtext, sel, m)
    RRC_DENSE[tag] = {q: {d: v[0][1] for d, v in rep[q].items()} for q in qids}
    print(f'[{tag}] chọn chunk {time.time()-t0:.0f}s | '
          f'{np.mean([len(v) for q in qids for v in rep[q].values()]):.2f} chunk/doc thực nhận',
          flush=True)

    txt   = dict(zip(qids, qtext))
    jobs  = [(q, txt[q], rep[q]) for q in qids]
    parts = [jobs[i::len(RR_DEVICES)] for i in range(len(RR_DEVICES))]
    store, lock, done = {}, threading.Lock(), [0]

    def work(dev, part):
        mdl = get_reranker(dev)
        for q, qt, chunk_of in part:
            pairs, owner = [], []
            for d, cs in chunk_of.items():
                for ci, _ in cs:
                    pairs.append([qt, CHUNK_TEXTS[ci]]); owner.append(d)
            per = defaultdict(list)
            if pairs:
                s = mdl.predict(pairs, batch_size=RERANK_BATCH, show_progress_bar=False)
                for d, v in zip(owner, np.asarray(s, dtype=np.float64)):
                    per[d].append(float(v))
            with lock:
                store[q] = dict(per)
                done[0] += 1
                if done[0] % 200 == 0:
                    print(f'  {done[0]}/{len(jobs)} query', flush=True)

    t0  = time.time()
    ths = [threading.Thread(target=work, args=(d, p), daemon=True)
           for d, p in zip(RR_DEVICES, parts) if p]
    for t in ths: t.start()
    for t in ths: t.join()

    RRC[tag], RRC_META[tag] = store, (top, m, maxlen)
    npair = sum(len(v) for q in store for v in store[q].values())
    v = np.array([s for q in store for ss in store[q].values() for s in ss])
    print(f'[{tag}] {npair} cặp | {(time.time()-t0)/60:.1f} phút', flush=True)
    print(f'[{tag}] điểm: min={v.min():.3f} p50={np.median(v):.3f} max={v.max():.3f} '
          f'std={v.std():.3f} | phân biệt {100*len(np.unique(np.round(v,4)))/len(v):.1f}%')
    if not np.isfinite(v).all():
        print('  !! có NaN/Inf -> đặt RERANK_FP16 = False rồi chạy lại')
    if v.std() < 1e-3:
        print('  !! điểm gần như hằng số -> reranker KHÔNG hoạt động, kiểm tra RERANKER')
    return tag

print('ok')


ok


In [19]:
#@title 7B.4a — pooling + fusion (chỉ định nghĩa hàm)
import numpy as np, pandas as pd

def pool_scores(v, mode='max', tau=0.5, lam=0.3):
    """Gộp điểm các chunk của MỘT doc thành điểm doc.

    'max' là mặc định cũ. Gold doc có p50 = 30 chunk và thường có 2-3 Điều liên
    quan -> 'lse' (max mềm) và 'top2' thưởng thêm cho doc có nhiều chunk khớp,
    trong khi vẫn không phạt doc dài chỉ có 1 Điều đúng như 'mean' làm.
    """
    v = np.asarray(v, float)
    if len(v) == 0:
        return -1e30
    if mode == 'max':
        return float(v.max())
    if mode == 'mean':
        return float(v.mean())
    if mode == 'top2':
        return float(v.max() + lam * (np.sort(v)[-2] if len(v) > 1 else v.max()))
    if mode == 'lse':
        mx = float(v.max())
        return mx + tau * float(np.log(np.exp((v - mx) / tau).sum()))
    raise ValueError(mode)

def _z(d):
    """z-norm trong PHẠM VI MỘT CÂU. Điểm cross-encoder không cùng thang giữa các
    câu hỏi, chuẩn hoá toàn cục là trộn hai phân bố khác nhau."""
    v = np.array(list(d.values()), float)
    mu, sd = v.mean(), v.std() + 1e-9
    return {k: (x - mu) / sd for k, x in d.items()}

def rank_from(tag, cand, pool='max', tau=0.5, lam=0.3, fuse='score',
              alpha=0.0, wc=1.0, wr=1.0, k=RRF_K):
    """-> {qid: [doc_id đã xếp hạng]}.

    fuse='score': z(rerank) + alpha*z(dense) — giữ được MARGIN điểm, thứ mà RRF
                  vứt bỏ. Với top-5 legal IR, margin là thông tin thật.
    fuse='rrf'  : RRF giữa thứ tự retriever và thứ tự rerank. Bản cũ ghim w=(1,1)
                  — đúng sai lầm mà Stage 6 đã học được với BM25.
    """
    rr, dn, out = RRC[tag], RRC_DENSE[tag], {}
    for q in rr:
        sel = [d for d in cand[q] if d in rr[q]]
        if not sel:
            out[q] = list(cand[q]); continue
        pooled = {d: pool_scores(rr[q][d], pool, tau, lam) for d in sel}
        if fuse == 'score':
            zr = _z(pooled)
            zd = _z({d: dn[q].get(d, 0.0) for d in sel}) if alpha else None
            key = (lambda d: -(zr[d] + alpha * zd[d])) if alpha else (lambda d: -zr[d])
            out[q] = sorted(sel, key=key) + [d for d in cand[q] if d not in rr[q]]
        else:
            s = {}
            for i, d in enumerate(cand[q]):
                s[d] = s.get(d, 0.0) + wc / (k + i + 1)
            for i, d in enumerate(sorted(sel, key=lambda d: -pooled[d])):
                s[d] = s.get(d, 0.0) + wr / (k + i + 1)
            out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])]
    return out

print('ok — rank_from(tag, cand, pool=..., fuse=..., ...)')


ok — rank_from(tag, cand, pool=..., fuse=..., ...)


---
## Stage 8 — Fine-tune bi-encoder *(ROI cao nhất sau reranker)*

6.000 cặp (câu hỏi → văn bản gold) trong train split là tài sản lớn nhất bạn có.
Embedding off-the-shelf hiểu tiếng Việt phổ thông nhưng **không** hiểu văn phong pháp quy.

Quy trình **iterative hard negative mining**:
1. Chạy retrieval hiện tại trên train split → top-50 doc.
2. Bỏ gold ra → phần còn lại là **hard negative** (giống gold nhưng sai — đúng thứ model
   cần học phân biệt; random negative gần như vô dụng).
3. Train `MultipleNegativesRankingLoss`.
4. Chạy lại cell này → vòng 2 mine negative bằng chính model vừa train, thường thêm
   được vài điểm nữa.

Cell này chạy **trọn gói**, không phải sửa cờ thủ công nữa:

```
mine negative (model hiện tại) → nhả VRAM → train → encode lại corpus bằng model mới
→ nạp lại query encoder → đo lại val (dense / hybrid / rerank)
```

Sau khi nó chạy xong, `MODEL_PATH` đã trỏ vào checkpoint fine-tune nên **Stage 9
(private test) tự dùng model mới** — cứ chạy tiếp các cell phía dưới.

Bật/tắt bằng `DO_FINETUNE` ở **Stage 0**.

> **VRAM.** Cell tự gọi `free_dense()` + xoá cache cross-encoder trước khi train, tự ép
> về **1 GPU** trong lúc `.fit()` (2 GPU + `DataParallel` vừa nhân đôi bản sao model trên
> `cuda:0` vừa cắt đôi in-batch negatives — thua cả về VRAM lẫn chất lượng), đóng băng
> bảng embedding từ vựng (~256M/568M param) và dùng `CachedMultipleNegativesRankingLoss`
> (GradCache). Nhờ GradCache, VRAM phụ thuộc `FT_MINIBATCH` chứ không phải `FT_BATCH`:
>
> * **OOM** → hạ `FT_MINIBATCH` (4 → 2 → 1), rồi mới đến `FT_GRAD_CKPT = True`.
>   **Đừng** hạ `FT_BATCH` — đó chính là số in-batch negative, hạ là train yếu đi.
> * Còn dư VRAM → tăng `FT_MINIBATCH` cho nhanh, hoặc tăng `FT_BATCH` cho mạnh.
>
> **Encode lại tốn thời gian bằng đúng Stage 5** (embedding của model cũ không dùng lại
> được). Vì vậy hãy train xong hẳn rồi mới encode lại — đừng encode lại sau mỗi epoch.


In [20]:
#@title Mine hard negatives + train + encode lại corpus
# ---- Tham số fine-tune -------------------------------------------------------
# FT_BATCH là số triplet trong MỘT bước optimizer = số in-batch negative của MNRL.
# Càng lớn càng tốt cho contrastive learning, nhưng nó KHÔNG còn là thứ quyết
# định VRAM nữa: CachedMultipleNegativesRankingLoss (GradCache) chỉ nạp
# FT_MINIBATCH câu lên GPU mỗi lượt forward/backward rồi ghép gradient lại, nên
# VRAM ~ FT_MINIBATCH chứ không phải FT_BATCH.
#   OOM  -> hạ FT_MINIBATCH (4 -> 2 -> 1), ĐỪNG hạ FT_BATCH.
#   Còn dư VRAM -> tăng FT_MINIBATCH cho nhanh, hoặc tăng FT_BATCH cho mạnh.
FT_BATCH     = 32
FT_MINIBATCH = 4
FT_MAXSEQ    = 512   # hạ so với MAX_SEQ để đổi lấy batch lớn hơn
FT_EPOCHS    = 2

# multilingual-e5-base có bảng token embedding lớn. Grad + 2 moment AdamW
# cho bảng đó tốn nhiều VRAM trong khi đóng góp fine-tune retrieval thường nhỏ,
# gần như không đóng góp gì cho retrieval văn bản pháp quy tiếng Việt.
FT_FREEZE_EMBEDDINGS = True
FT_GRAD_CKPT         = False   # bật nếu vẫn OOM ở FT_MINIBATCH=1 (chậm ~30%)

DO_FINETUNE = globals().get('DO_FINETUNE', True)   # đặt ở Stage 0

# Chống phân mảnh allocator (đúng thứ mà thông báo OOM của torch gợi ý).
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

if DO_FINETUNE:
    import random, shutil, gc, os, glob, time, torch, numpy as np
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader

    # ---- 1. mine hard negative bằng retrieval HIỆN TẠI -----------------------
    # Chạy lần 2 (sau khi đã có checkpoint) thì retrieval ở đây chính là model
    # fine-tune vòng 1 -> negative khó hơn -> đúng nghĩa "iterative mining".
    tr_qids = TRAIN_QIDS
    tr_text = [QUESTIONS[q] for q in tr_qids]
    print('mine hard negatives...', flush=True)
    cand = rrf(retrieve_bm25(tr_qids, tr_text), retrieve_dense(tr_qids, tr_text),
               weights=RRF_WEIGHTS)
    rep  = best_chunk_per_doc(tr_qids, tr_text, cand)

    # chunk đại diện của chính gold doc, dùng làm positive
    gold_rep = best_chunk_per_doc(tr_qids, tr_text, {q: list(GOLD[q]) for q in tr_qids})

    rng = random.Random(SEED)
    examples = []
    for q in tr_qids:
        gold = GOLD[q]
        pos_ci = [gold_rep[q][g] for g in gold if g in gold_rep[q]]
        if not pos_ci:
            continue
        negs = [d for d in cand[q][:50] if d not in gold]
        if not negs:
            continue
        for nd in rng.sample(negs, min(4, len(negs))):
            examples.append(InputExample(texts=[format_dense_queries([QUESTIONS[q]])[0],
                                                format_dense_passages([CHUNK_TEXTS[rng.choice(pos_ci)]])[0],
                                                format_dense_passages([CHUNK_TEXTS[rep[q][nd]]])[0]]))
    print(f'{len(examples)} training triplet')
    assert examples, 'không mine được triplet nào — kiểm tra GOLD / TRAIN_QIDS'

    # ---- 2. nhả VRAM trước khi train ----------------------------------------
    # EMB_T (corpus embedding), query encoder và các cross-encoder đang giữ vài GB.
    # Không nhả thì train là OOM ngay. EMB (numpy) cũng bỏ luôn vì lát nữa encode
    # lại từ đầu bằng model mới. empty_cache() CHỈ tác dụng lên device hiện tại
    # -> phải quét hết mọi GPU, không thì cache của cuda:1 vẫn nằm nguyên đó.
    free_dense()
    RERANKERS.clear()
    _RR_SCORE.clear()   # candidate đổi hết sau khi fine-tune -> cache cũ vô nghĩa
    EMB = None
    gc.collect()
    if torch.cuda.is_available():
        for _i in range(torch.cuda.device_count()):
            with torch.cuda.device(_i):
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
        for _i in range(torch.cuda.device_count()):
            print('cuda:%d còn giữ %.2f GB sau khi dọn'
                  % (_i, torch.cuda.memory_allocated(_i) / 1e9), flush=True)

    # ---- 3. train ------------------------------------------------------------
    # Luôn train từ checkpoint GỐC, không train chồng lên bản fine-tune trước:
    # train chồng nhiều vòng trên cùng 6k câu hỏi là đường thẳng tới overfit.
    t0 = time.time()
    model = SentenceTransformer(BIENCODER, device=DEV)
    model.max_seq_length = FT_MAXSEQ

    auto = model[0].auto_model
    if FT_FREEZE_EMBEDDINGS:
        n_frozen = 0
        for p in auto.get_input_embeddings().parameters():
            p.requires_grad_(False)
            n_frozen += p.numel()
        print('đóng băng %.0fM param embedding (tiết kiệm ~%.1f GB grad + AdamW state)'
              % (n_frozen / 1e6, n_frozen * 12 / 1e9))
    if FT_GRAD_CKPT:
        auto.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={'use_reentrant': False})
        print('gradient checkpointing: ON')

    # GradCache: batch lớn với VRAM của batch nhỏ. Bản ST quá cũ không có loss
    # này thì rơi về MNRL thường + tự hạ FT_BATCH cho khớp VRAM.
    if hasattr(losses, 'CachedMultipleNegativesRankingLoss'):
        loss = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=FT_MINIBATCH)
        print(f'loss = CachedMNRL(batch={FT_BATCH}, mini_batch={FT_MINIBATCH})')
    else:
        FT_BATCH = min(FT_BATCH, 8)
        loss = losses.MultipleNegativesRankingLoss(model)
        print(f'ST cũ, không có CachedMNRL -> MNRL thường, hạ FT_BATCH={FT_BATCH}')

    dl = DataLoader(examples, shuffle=True, batch_size=FT_BATCH, drop_last=True)

    # BẮT BUỘC 1 GPU cho bước này. .fit() đi qua HF Trainer; Trainer thấy
    # device_count() > 1 là tự bọc nn.DataParallel, mà DP vừa replicate toàn bộ
    # model lên cuda:0 mỗi step (thêm ~2.3 GB nằm cạnh optimizer state -> chính
    # là OOM vừa gặp) vừa cắt batch làm hai khiến MNRL mất một nửa in-batch
    # negative. Ép device_count() = 1 đúng lúc gọi fit() rồi trả lại ngay, để
    # bước encode corpus phía dưới vẫn dùng được cả hai GPU.
    _real_device_count = torch.cuda.device_count
    torch.cuda.device_count = lambda: 1
    try:
        model.fit(train_objectives=[(dl, loss)],
                  epochs=FT_EPOCHS, warmup_steps=int(0.1 * len(dl) * FT_EPOCHS),
                  use_amp=True, output_path=FT_BIENCODER_PATH, show_progress_bar=True)
    finally:
        torch.cuda.device_count = _real_device_count

    if torch.cuda.is_available():
        print('VRAM đỉnh lúc train: %.2f GB' % (torch.cuda.max_memory_allocated(0) / 1e9))
    print('train %.1f phút | đã lưu -> %s' % ((time.time() - t0) / 60, FT_BIENCODER_PATH))
    del model, dl, loss, examples, auto
    gc.collect()
    if torch.cuda.is_available():
        for _i in range(torch.cuda.device_count()):
            with torch.cuda.device(_i):
                torch.cuda.empty_cache()

    # ---- 4. encode lại corpus bằng model mới --------------------------------
    USE_FINETUNED = True
    set_dense_model(FT_BIENCODER_PATH)
    if glob.glob(f'{EMB_DIR}/s_*.npy'):
        # Checkpoint fine-tune luôn ghi vào cùng một path -> TAG không đổi -> shard
        # còn lại là của LẦN TRAIN TRƯỚC. Dùng lại là đang search bằng embedding của
        # model cũ trong khi query đi qua model mới: sai lệch mà không báo lỗi.
        print('xoá embedding của lần fine-tune trước:', EMB_DIR, flush=True)
        shutil.rmtree(EMB_DIR, ignore_errors=True)
        os.makedirs(EMB_DIR, exist_ok=True)
    EMB_DIRS = [EMB_DIR]      # cache chỉ-đọc cũng là của model cũ -> không đọc
    t0  = time.time()
    EMB = encode_corpus()
    assert len(EMB) == len(chunks), f'emb {len(EMB)} != chunks {len(chunks)}'
    print('EMB', EMB.shape, '| %.1f MB | %.1f phút' % (EMB.nbytes / 1e6, (time.time() - t0) / 60))
    load_query_model()

    # ---- 5. đo lại trên val --------------------------------------------------
    # Ghi đè dense_val / hybrid_val / rerank_val để Stage 9 và bảng thí nghiệm
    # nói về cùng một model.
    dense_val  = retrieve_dense(VAL_QIDS, val_text)
    m = show('[ft] dense', dense_val)
    log_exp('dense_ft', m, FT_BIENCODER_PATH)

    hybrid_val = rrf(bm25_val, dense_val, weights=RRF_WEIGHTS)
    m_none = show('[ft] hybrid RRF', hybrid_val)
    log_exp('hybrid_rrf_ft', m_none, f'K={RRF_K}')

    print('[production] bỏ benchmark reranker zero-shot sau Stage 8; '
          'không ảnh hưởng model hay submission.')
    print('MODEL_PATH giờ là bản fine-tune -> Stage 8B dùng dense candidates mới.')
else:
    print('DO_FINETUNE=False — bỏ qua (Stage 9 sẽ dùng', MODEL_PATH, ').')


DO_FINETUNE=False — bỏ qua (Stage 9 sẽ dùng /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/models/biencoder-ft-v3b_ft_reranker_improved ).


---
## Stage 8B — Fine-tune reranker *(việc ăn điểm chính của bản B)*

Chẩn đoán từ log v2: `recall@100 = 0.9897` nhưng `recall@5 = 0.8996`, và reranker zero-shot
**không vượt nổi dense đơn thuần**. Retriever đã xong việc; toàn bộ dư địa nằm ở xếp hạng
trong top-30 (trần ≈ 0.975). `Vietnamese_Reranker` được train trên QA/retrieval tổng quát,
không phải văn phong pháp quy → đây là domain adaptation, không phải thiếu dữ liệu.

### Ba chỗ khác Stage 8 cũ

**1. Negative lấy từ đúng candidate generator của inference.** Stage 8 cũ mine từ
`rrf(bm25, dense)` trong khi pipeline chạy `base='dense'` → reranker học phân biệt loại
negative mà lúc chạy thật nó **không bao giờ gặp**. Ở đây negative = `dense` top-`FT_RR_TOP`,
đúng cái sẽ đưa cho nó lúc inference.

**2. Chunk đại diện chọn bằng đúng `select_chunks` của inference.** Train trên chunk chọn
kiểu A rồi suy luận trên chunk chọn kiểu B là lệch phân bố.

**3. Listwise loss.** `LambdaLoss` (NDCGLoss2++) chấm cả nhóm 1 pos + `FT_RR_NNEG` neg cùng
lúc, tối ưu trực tiếp thứ tự — đúng thứ metric đo. Pointwise BCE chỉ học "liên quan/không",
không học *xếp ai trên ai*.

### VRAM trên T4 15GB

XLM-R large 560M param, trong đó bảng embedding 256M. Đóng băng bảng đó (giống Stage 8 làm
với bi-encoder) còn ~304M trainable → AdamW state ~2.4GB, tổng ~8–9GB. Cộng gradient
checkpointing là vừa. **Ép 1 GPU** trong lúc train bằng đúng cơ chế Stage 8 dùng: HF Trainer
thấy `device_count() > 1` là tự bọc `DataParallel`, vừa nhân đôi model trên `cuda:0` vừa cắt
nhóm listwise làm hai — thua cả VRAM lẫn chất lượng.

> **`SMOKE = True` là mặc định.** Nó chạy 300 câu / 30 bước để xác nhận pipeline không sập,
> mất ~5 phút. Xem log thấy loss giảm thì đặt `SMOKE = False` và chạy lại cell. Đừng đốt 3h
> GPU vào một cell chưa bao giờ chạy hết.


In [21]:
#@title 8B.1 — Sinh dữ liệu train listwise
FT_RR_TOP    = 40     # = RERANK_TOP lúc inference: negative phải cùng phân bố
FT_RR_NNEG   = 11      # 1 pos : 7 neg -> nhóm 8 doc/câu
FT_RR_MAXLEN = 768    # khớp config val tốt nhất cũ (top30 × m8 × L768)
FT_RR_EPOCHS = 1
FT_RR_LR     = 1e-5
FT_RR_ACCUM  = 8      # nhóm/bước optimizer (VRAM ~ per_device_batch, không phải cái này)
FT_RR_FREEZE_EMB = True
FT_RR_GRAD_CKPT  = True

# ĐỌC KỸ: True = 300 câu / 30 bước (~5 phút) để kiểm tra cell 8B.2 chạy được.
# Thấy loss giảm thì đặt False rồi chạy lại 8B.1 + 8B.2. Đừng đốt 3h GPU vào một
# cell chưa bao giờ chạy tới dòng cuối.
SMOKE = False  #@param {type:'boolean'}

import numpy as np, random, time

if RUN_PHASE == 1:
    _tr = TRAIN_QIDS[:300] if SMOKE else TRAIN_QIDS
    _tt = [QUESTIONS[q] for q in _tr]
    print(f'{len(_tr)} câu train' + ('  (SMOKE)' if SMOKE else ''), flush=True)

    t0 = time.time()
    _cand = retrieve_dense(_tr, _tt)                       # đúng generator của inference
    print('dense candidates: %.1f phút' % ((time.time() - t0) / 60), flush=True)

    # 1 chunk/doc là đủ cho train: lúc inference điểm doc = MAX trên m chunk, nên
    # chunk kém của gold doc không làm hỏng gì. Positive = chunk tốt nhất của gold doc.
    _neg_rep = select_chunks(_tr, _tt, {q: _cand[q][:FT_RR_TOP] for q in _tr}, 1)
    _pos_rep = select_chunks(_tr, _tt, {q: list(GOLD[q]) for q in _tr}, 3)

    _rng, rows, n_skip = random.Random(SEED), [], 0
    for q in _tr:
        gold = GOLD[q]
        # Keep up to two positive evidence chunks. For a multi-answer query,
        # positives may come from different gold documents; for a single-answer
        # query, they are two high-scoring chunks from the same gold document.
        pos = []
        for g in sorted(gold):
            for _ci, _s in _pos_rep[q].get(g, [])[:3]:
                if int(_ci) not in pos:
                    pos.append(int(_ci))
                if len(pos) >= 2:
                    break
            if len(pos) >= 2:
                break
        negs = [d for d in _cand[q][:FT_RR_TOP]
                if d not in gold and d in _neg_rep[q]]
        if not pos or len(negs) < 2:
            n_skip += 1; continue
        negs = _rng.sample(negs, min(FT_RR_NNEG, len(negs)))
        docs = ([CHUNK_TEXTS[_ci] for _ci in pos]
                + [CHUNK_TEXTS[_neg_rep[q][d][0][0]] for d in negs])
        rows.append({'query': QUESTIONS[q],
                     'docs': docs,
                     'labels': [1.0] * len(pos) + [0.0] * len(negs)})
    print(f'{len(rows)} nhóm listwise | bỏ {n_skip} câu (không có gold trong corpus hoặc <2 neg)')
    assert rows, 'không sinh được dữ liệu train — kiểm tra GOLD / TRAIN_QIDS'
    print('ví dụ: %d doc/nhóm | query: %s' % (len(rows[0]['docs']), rows[0]['query'][:80]))
else:
    rows = None
    print('RUN_PHASE=2: bỏ sinh train set; sẽ load checkpoint từ output Version 1.')


RUN_PHASE=2: bỏ sinh train set; sẽ load checkpoint từ output Version 1.


In [22]:
#@title 8B.2 — Train cross-encoder (LambdaLoss, 1 GPU, freeze embeddings)
import os, gc, time, torch
os.environ['WANDB_DISABLED'] = 'true'          # Kaggle: không có wandb -> tránh prompt
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

try:
    from datasets import Dataset
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
    from datasets import Dataset

# API trainer cho cross-encoder chỉ có từ sentence-transformers 4.x trở lên.
# Stage 0 ghim 5.4.0 nên phải có; thiếu là môi trường không đúng -> báo rõ, đừng
# lặng lẽ rơi về .fit() cũ (loss listwise không chạy được qua API đó).
try:
    from sentence_transformers.cross_encoder import (
        CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments)
    from sentence_transformers.cross_encoder import losses as ce_losses
except ImportError as e:
    import sentence_transformers as _st
    raise ImportError(
        f'sentence-transformers {_st.__version__} không có CrossEncoderTrainer '
        f'({e}). Chạy lại cell Stage 0 "Mount Drive + cài dependency" để cài '
        f'bản {ST_WANT}, rồi Restart runtime.') from e

FT_RERANKER_REL = f'models/reranker-ft-{PROFILE}'
FT_RERANKER_PATH = f'{ART}/{FT_RERANKER_REL}'

if RUN_PHASE == 1:
    # ---- nhả VRAM trước khi train ------------------------------------------------
    # EMB_T (corpus embedding), query encoder và các cross-encoder inference đang giữ
    # vài GB. Không nhả thì train OOM ngay. EMB (numpy) GIỮ LẠI — model dense không
    # đổi nên không phải encode lại, chỉ cần load_query_model() sau khi train xong.
    free_dense()
    RERANKERS.clear()
    if '_RR_SCORE' in globals():
        _RR_SCORE.clear()
    gc.collect()
    if torch.cuda.is_available():
        for _i in range(torch.cuda.device_count()):
            with torch.cuda.device(_i):
                torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        for _i in range(torch.cuda.device_count()):
            print('cuda:%d còn giữ %.2f GB' % (_i, torch.cuda.memory_allocated(_i) / 1e9))

    ds = Dataset.from_list(rows)
    print(ds)

    model = CrossEncoder(RERANKER, num_labels=1, max_length=FT_RR_MAXLEN, device=DEV)

    if FT_RR_FREEZE_EMB:
        _n = 0
        for p in model.model.get_input_embeddings().parameters():
            p.requires_grad_(False); _n += p.numel()
        print('đóng băng %.0fM param embedding (~%.1f GB grad + AdamW state)'
              % (_n / 1e6, _n * 12 / 1e9))

    loss = ce_losses.LambdaLoss(model)
    print('loss =', type(loss).__name__)

    targs = CrossEncoderTrainingArguments(
        output_dir=FT_RERANKER_PATH,
        num_train_epochs=1,
        per_device_train_batch_size=1,            # 1 nhóm = 8 sequence -> VRAM phụ thuộc cái này
        gradient_accumulation_steps=FT_RR_ACCUM,
        learning_rate=FT_RR_LR,
        # Transformers v5: float warmup_steps in [0, 1) is interpreted as a ratio.
        warmup_steps=0.1,
        fp16=torch.cuda.is_available(),
        gradient_checkpointing=FT_RR_GRAD_CKPT,
        # use_reentrant=False BẮT BUỘC khi đã đóng băng bảng embedding: với bản
        # reentrant, block đầu nhận input không requires_grad và gradient bị cắt
        # ngay tại đó -> train chạy êm nhưng model gần như không học.
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=25,
        save_strategy='steps',
        save_steps=0.2,                           # 5 mốc đều trên toàn bộ training run
        save_total_limit=5,
        save_only_model=True,                     # tránh optimizer state làm vượt 20GB Kaggle
        report_to=[],
        seed=SEED,
        max_steps=30 if SMOKE else -1,
    )
    # HF Trainer thấy device_count() > 1 là tự bọc DataParallel: nhân đôi model trên
    # cuda:0 (cạnh optimizer state -> OOM) và cắt nhóm listwise làm hai (mất một nửa
    # negative của LambdaLoss). Ép về 1 GPU đúng lúc train rồi trả lại ngay, để bước
    # rerank phía sau vẫn dùng được cả hai GPU. Cùng cơ chế Stage 8 dùng.
    # TrainingArguments already cached device_count during construction, so patching
    # torch.cuda.device_count here is too late. Force its cache to one GPU: LambdaLoss
    # needs CrossEncoder.preprocess(), which torch.nn.DataParallel does not expose.
    targs._n_gpu = 1
    t0 = time.time()
    CrossEncoderTrainer(model=model, args=targs, train_dataset=ds, loss=loss).train()

    # Checkpoint 100% cuối đã được Trainer lưu; không tạo thêm /final trùng 2-3GB.
    if torch.cuda.is_available():
        print('VRAM đỉnh: %.2f GB' % (torch.cuda.max_memory_allocated(0) / 1e9))
    print('train %.1f phút -> %s (5 checkpoint)' % ((time.time() - t0) / 60, FT_RERANKER_PATH))

    del model, loss
    gc.collect()
    if torch.cuda.is_available():
        for _i in range(torch.cuda.device_count()):
            with torch.cuda.device(_i):
                torch.cuda.empty_cache()

    if SMOKE:
        print('\n>>> SMOKE xong. Loss có giảm không? Nếu có: đặt SMOKE = False ở 8B.1,'
              ' chạy lại 8B.1 rồi 8B.2. Nếu không: dừng lại, đừng chạy full.')
else:
    FT_RERANKER_PATH = cache_path(FT_RERANKER_REL)
    assert FT_RERANKER_PATH and os.path.isdir(FT_RERANKER_PATH), (
        'Không thấy checkpoint Version 1 trong dataset v3b-finetune-reranker-improve. '
        'của Version 1, rồi chạy lại với RUN_PHASE=2.')
    print('RUN_PHASE=2: load reranker checkpoint <-', FT_RERANKER_PATH)


RUN_PHASE=2: load reranker checkpoint <- /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/models/reranker-ft-v3b_ft_reranker_improved


In [23]:
if RUN_PHASE == 2:
    #@title 8B.3 — Chọn checkpoint hai tầng: 300 câu shortlist → 1.000 câu final
    import glob, os, numpy as np

    load_query_model()
    BASE_CAND = BASES[RERANK_BASE] if 'BASES' in globals() else dense_val
    _paths = [(os.path.basename(p), p)
              for p in sorted(glob.glob(f'{FT_RERANKER_PATH}/checkpoint-*'))]
    if not _paths and os.path.isdir(f'{FT_RERANKER_PATH}/final'):
        _paths.append(('final', f'{FT_RERANKER_PATH}/final'))
    assert _paths, 'Không tìm thấy checkpoint reranker trong output Version 1.'

    # Cố định theo VAL_QIDS (đã cố định bởi SEED), không random lại giữa các run.
    SCREEN_N = min(300, len(VAL_QIDS))
    screen_qids = VAL_QIDS[:SCREEN_N]
    screen_text = [QUESTIONS[q] for q in screen_qids]
    screen_base = {q: BASE_CAND[q] for q in screen_qids}
    screen_gold = {q: GOLD[q] for q in screen_qids}

    screen_metric = {}
    for _label, _path in _paths:
        RERANKER = _path
        RERANKERS.clear()
        for _d in RR_DEVICES:
            get_reranker(_d)
        _tag = 'screen_' + _label
        rerank_collect(screen_qids, screen_text, screen_base, RERANK_TOP,
                       RERANK_CHUNKS_PER_DOC, FT_RR_MAXLEN, tag=_tag)
        _p = rank_from(_tag, screen_base, pool='max', fuse='score', alpha=0.0)
        screen_metric[_label] = evaluate(_p, screen_gold)
        print(f"[screen {_label}] recall@5={screen_metric[_label]['recall@5']:.4f}")

    # Chỉ model đã train; zero-shot đã được baseline đo và không cần tốn thêm 45 phút.
    shortlist = sorted(screen_metric,
                       key=lambda x: (screen_metric[x]['OFFICIAL_recall'],
                                      screen_metric[x]['OFFICIAL_precision']),
                       reverse=True)[:2]
    print('shortlist full-val:', shortlist)

    # Không để tag 300 câu lọt vào sweep full-val 1.000 câu ở cell 7B.4b.
    for _label in list(screen_metric):
        _tag = 'screen_' + _label
        RRC.pop(_tag, None); RRC_DENSE.pop(_tag, None); RRC_META.pop(_tag, None)

    preds_of, metric_of, full_tag_of = {}, {}, {}
    for _label in shortlist:
        _path = dict(_paths)[_label]
        RERANKER = _path
        RERANKERS.clear()
        for _d in RR_DEVICES:
            get_reranker(_d)
        _tag = 'full_' + _label
        rerank_collect(VAL_QIDS, val_text, BASE_CAND, RERANK_TOP,
                       RERANK_CHUNKS_PER_DOC, FT_RR_MAXLEN, tag=_tag)
        _p = rank_from(_tag, BASE_CAND, pool='max', fuse='score', alpha=0.0)
        preds_of[_label] = _p
        metric_of[_label] = show(f'[{_label}] full-val max/score', _p)
        full_tag_of[_label] = _tag
        log_exp(f'rerank_ft_{_label}', metric_of[_label],
                f'{_path} raw_logits maxlen={FT_RR_MAXLEN}')

    _best_label = max(metric_of, key=lambda x: (metric_of[x]['OFFICIAL_recall'],
                                                 metric_of[x]['OFFICIAL_precision']))
    RERANKER = dict(_paths)[_best_label]
    RERANKERS.clear()
    for _d in RR_DEVICES:
        get_reranker(_d)
    print('\nRERANKER tốt nhất ->', RERANKER)
    compare(f'ft:{_best_label}', preds_of[_best_label], 'dense đơn thuần', dense_val)

    # Tái dùng full-val raw scores cho đúng variant 30×8×768 ở grid kế tiếp.
    _reuse = full_tag_of[_best_label]
    RRC['ft_top30_m8_L768'] = RRC[_reuse]
    RRC_DENSE['ft_top30_m8_L768'] = RRC_DENSE[_reuse]
    RRC_META['ft_top30_m8_L768'] = RRC_META[_reuse]
else:
    print('RUN_PHASE=1: bỏ qua — bước chọn checkpoint thuộc Version 2.')


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[screen_checkpoint-150] chọn chunk 1s | 7.53 chunk/doc thực nhận
  200/300 query
[screen_checkpoint-150] 67763 cặp | 15.3 phút
[screen_checkpoint-150] điểm: min=-11.438 p50=-10.500 max=11.398 std=2.588 | phân biệt 5.4%
[screen checkpoint-150] recall@5=0.9294


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[full_checkpoint-750] chọn chunk 2s | 7.52 chunk/doc thực nhận
  200/1000 query
  400/1000 query
  600/1000 query
  800/1000 query
  1000/1000 query
[full_checkpoint-750] 225690 cặp | 51.8 phút
[full_checkpoint-750] điểm: min=-11.430 p50=-11.172 max=11.391 std=2.673 | phân biệt 2.9%
[checkpoint-750] full-val max/score recall@5=0.9529 recall@10=0.9690 recall@20=0.9799 recall@30=0.9827 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9529 OFFICIAL_precision=0.2052


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


RERANKER tốt nhất -> /kaggle/input/datasets/kitnehi1211/v3b-finetune-reranker-improve/DSC2026_Task1/artifacts/models/reranker-ft-v3b_ft_reranker_improved/checkpoint-750
  ft:checkpoint-750  vs  dense đơn thuần
    delta=+0.0246  CI95=[+0.0128, +0.0369]  P(A>B)=1.000
    thắng 38 / thua 13 / bằng 949 câu  ->  THẮNG rõ


In [24]:
if RUN_PHASE == 2:
    #@title 7B.3b — Grid hình dạng input, CHẠY LẠI với reranker đã fine-tune
    # Trọng số/pooling tối ưu của reranker gốc không còn đúng cho model đã fine-tune,
    # và hình dạng input tối ưu cũng có thể đổi. Model improved học ở maxlen=768; L768 được reuse từ full-val chọn checkpoint.
    # Hai biến thể L512 kiểm tra trade-off ngữ cảnh/tốc độ mà không train lại.
    BASE_CAND = BASES[RERANK_BASE]
    VARIANTS  = [(30, 8, 512), (20, 8, 512), (30, 8, 768)]  # L768 tự reuse

    for _top, _m, _L in VARIANTS:
        rerank_collect(VAL_QIDS, val_text, BASE_CAND, _top, _m, _L, tag=f'ft_top{_top}_m{_m}_L{_L}')
    print('\nRRC:', list(RRC))
else:
    print('RUN_PHASE=1: bỏ qua — bước này thuộc Version 2.')


[ft_top30_m8_L512] chọn chunk 2s | 7.52 chunk/doc thực nhận
  200/1000 query
  400/1000 query
  600/1000 query
  800/1000 query
  1000/1000 query
[ft_top30_m8_L512] 225690 cặp | 37.6 phút
[ft_top30_m8_L512] điểm: min=-11.430 p50=-11.281 max=11.391 std=2.500 | phân biệt 2.7%
[ft_top20_m8_L512] chọn chunk 2s | 7.54 chunk/doc thực nhận
  200/1000 query
  400/1000 query
  600/1000 query
  800/1000 query
  1000/1000 query
[ft_top20_m8_L512] 150745 cặp | 25.2 phút
[ft_top20_m8_L512] điểm: min=-11.430 p50=-11.070 max=11.391 std=2.850 | phân biệt 3.9%
[ft_top30_m8_L768] đã có trong RRC -> bỏ qua

RRC: ['full_checkpoint-600', 'full_checkpoint-750', 'ft_top30_m8_L768', 'ft_top30_m8_L512', 'ft_top20_m8_L512']


In [25]:
if RUN_PHASE == 2:
    #@title 7B.4b — Sweep 180 tổ hợp (CPU, 0 GPU) rồi chốt config
    # Chạy trên điểm chunk-level đã lưu trong RRC -> không tốn GPU. ~1-2 phút CPU.
    import numpy as np, pandas as pd

    assert RRC, 'RRC rỗng — chạy cell grid (7B.3b) trước.'

    GOLD_VAL = {q: GOLD[q] for q in VAL_QIDS}
    POOLS = [dict(pool='max'), dict(pool='lse', tau=0.25), dict(pool='lse', tau=0.5),
             dict(pool='top2', lam=0.2), dict(pool='top2', lam=0.4), dict(pool='mean')]
    FUSES = ([dict(fuse='score', alpha=a) for a in (0.0, 0.15, 0.3, 0.5, 0.8)] +
             [dict(fuse='rrf', wc=wc, wr=wr) for wc, wr in
              ((1, 1), (1, 2), (1, 3), (0.5, 1), (1, 0.5))])

    rows = []
    for tag in RRC:
        for p in POOLS:
            for f in FUSES:
                kw = dict(pool='max', tau=0.5, lam=0.3, fuse='score',
                          alpha=0.0, wc=1.0, wr=1.0)
                kw.update(p); kw.update(f)
                m = evaluate(rank_from(tag, BASE_CAND, **kw), GOLD_VAL)
                rows.append({'tag': tag, **kw, 'recall@5': m['recall@5'],
                             'OFFICIAL_recall': m['OFFICIAL_recall'],
                             'OFFICIAL_precision': m['OFFICIAL_precision']})
    print(f'{len(rows)} tổ hợp đã đo (CPU)')

    res = (pd.DataFrame(rows)
           .sort_values(['OFFICIAL_recall', 'OFFICIAL_precision'], ascending=False)
           .reset_index(drop=True))
    display(res.head(15)[['tag', 'pool', 'tau', 'lam', 'fuse', 'alpha', 'wc', 'wr',
                          'recall@5', 'OFFICIAL_recall']])

    # Bảng phụ: mỗi tag tốt nhất được bao nhiêu -> tách tác động của HÌNH DẠNG INPUT
    # ra khỏi tác động của pooling/fusion.
    print('\ntốt nhất theo từng tag (hình dạng input):')
    display(res.groupby('tag')['OFFICIAL_recall'].max().sort_values(ascending=False))

    # ---- Chốt config + kiểm định ghép cặp ---------------------------------------
    _b = res.iloc[0]
    BEST = {'tag': _b.tag, 'pool': _b.pool, 'tau': float(_b.tau), 'lam': float(_b.lam),
            'fuse': _b.fuse, 'alpha': float(_b.alpha), 'wc': float(_b.wc), 'wr': float(_b.wr),
            'top': RRC_META[_b.tag][0], 'm': RRC_META[_b.tag][1],
            'maxlen': RRC_META[_b.tag][2]}
    print('\nBEST =', BEST)
    best_preds = rank_from(BEST['tag'], BASE_CAND, BEST['pool'], BEST['tau'], BEST['lam'],
                           BEST['fuse'], BEST['alpha'], BEST['wc'], BEST['wr'])

    print('\n--- kiểm định ghép cặp (chỉ tin khi P(A>B) > 0.95) ---')
    compare('7B best', best_preds, 'dense đơn thuần', dense_val)
    if 'rerank_rrf_val' in globals():
        compare('7B best', best_preds, 'rerank cũ (selector cũ, RRF w=1,1)', rerank_rrf_val)
    if 'rerank_val' in globals():
        compare('7B best', best_preds, 'rerank cũ (replace)', rerank_val)

    m = show('7B best', best_preds)
    log_exp('7B_best', m, f"{BEST}")

    # ---- Trần còn lại -----------------------------------------------------------
    _bm    = evaluate(BASE_CAND, GOLD_VAL)
    _ceil  = _bm.get(f"recall@{BEST['top']}", _bm['recall@100'])
    print(f"\ntrần của rerank (recall@{BEST['top']} của base '{RERANK_BASE}') = {_ceil:.4f}"
          f"  |  đang đạt {m['recall@5']:.4f}  |  còn {100*(_ceil-m['recall@5']):.1f} điểm")
    print("nếu 7B best KHÔNG thắng dense rõ ràng -> domain mismatch đã được xác nhận,"
          " việc tiếp theo là fine-tune reranker (bản B), không phải thêm kênh retrieval.")
else:
    print('RUN_PHASE=1: bỏ qua — bước này thuộc Version 2.')


300 tổ hợp đã đo (CPU)


,tag,pool,tau,lam,fuse,alpha,wc,wr,recall@5,OFFICIAL_recall
0,full_checkpoint-750,top2,0.50,0.2,rrf,0.00,1.0,3.0,0.95850,0.95850
1,ft_top30_m8_L768,top2,0.50,0.2,rrf,0.00,1.0,3.0,0.95850,0.95850
2,full_checkpoint-750,top2,0.50,0.2,score,0.15,1.0,1.0,0.95800,0.95800
3,ft_top30_m8_L768,top2,0.50,0.2,score,0.15,1.0,1.0,0.95800,0.95800
4,full_checkpoint-750,max,0.50,0.3,rrf,0.00,1.0,2.0,0.95775,0.95775
5,full_checkpoint-750,max,0.50,0.3,rrf,0.00,0.5,1.0,0.95775,0.95775
6,full_checkpoint-750,lse,0.25,0.3,rrf,0.00,1.0,2.0,0.95775,0.95775
7,full_checkpoint-750,lse,0.25,0.3,rrf,0.00,0.5,1.0,0.95775,0.95775
8,full_checkpoint-750,lse,0.50,0.3,rrf,0.00,1.0,2.0,0.95775,0.95775
9,full_checkpoint-750,lse,0.50,0.3,rrf,0.00,0.5,1.0,0.95775,0.95775



tốt nhất theo từng tag (hình dạng input):


tag
ft_top30_m8_L768       0.95850
full_checkpoint-750    0.95850
full_checkpoint-600    0.95675
ft_top20_m8_L512       0.95525
ft_top30_m8_L512       0.95425
Name: OFFICIAL_recall, dtype: float64


BEST = {'tag': 'full_checkpoint-750', 'pool': 'top2', 'tau': 0.5, 'lam': 0.2, 'fuse': 'rrf', 'alpha': 0.0, 'wc': 1.0, 'wr': 3.0, 'top': 30, 'm': 8, 'maxlen': 768}

--- kiểm định ghép cặp (chỉ tin khi P(A>B) > 0.95) ---
  7B best  vs  dense đơn thuần
    delta=+0.0302  CI95=[+0.0194, +0.0419]  P(A>B)=1.000
    thắng 38 / thua 5 / bằng 957 câu  ->  THẮNG rõ
7B best                      recall@5=0.9585 recall@10=0.9711 recall@20=0.9809 recall@30=0.9827 recall@50=0.9849 recall@100=0.9879 OFFICIAL_recall=0.9585 OFFICIAL_precision=0.2068

trần của rerank (recall@30 của base 'dense') = 0.9827  |  đang đạt 0.9585  |  còn 2.4 điểm
nếu 7B best KHÔNG thắng dense rõ ràng -> domain mismatch đã được xác nhận, việc tiếp theo là fine-tune reranker (bản B), không phải thêm kênh retrieval.


---
## Stage 9 — Dự đoán private test + validate + đóng gói

Validator chặn đúng những lỗi làm mất điểm oan:
- Thiếu qid so với `private-official-legalr.json`
- Câu có **> 5** id → BTC cho câu đó **0 điểm cả Recall lẫn Precision**
- Câu có **< 5** id → tự bỏ điểm Recall vô cớ (Recall là metric chính, Precision chỉ tie-break)
- id trùng lặp trong một câu, id không phải string, id không tồn tại trong corpus
- `submission.json` bị nằm trong thư mục con của zip (lỗi rất hay gặp → rớt bài nộp)

In [26]:
if RUN_PHASE == 2:
    #@title 9B — full_pipeline dùng config đã chốt ở 7B.4
    # Ghi đè full_pipeline của Stage 9 để bài nộp đi qua ĐÚNG đường đã đo trên val.
    # Không sửa cell Stage 9 phía dưới — nó gọi full_pipeline() nên tự dùng bản này.
    def full_pipeline_v3(qids, qtext):
        qids = list(qids)
        d    = retrieve_dense(qids, qtext)
        cand = d if RERANK_BASE == 'dense' else \
            rrf(retrieve_bm25(qids, qtext), d, weights=RRF_WEIGHTS)
        tag = 'private_' + BEST['tag']
        rerank_collect(qids, qtext, cand, BEST['top'], BEST['m'], BEST['maxlen'], tag=tag)
        return rank_from(tag, cand, BEST['pool'], BEST['tau'], BEST['lam'],
                         BEST['fuse'], BEST['alpha'], BEST['wc'], BEST['wr'])

    full_pipeline = full_pipeline_v3
    print('full_pipeline -> v3 |', BEST)
else:
    print('RUN_PHASE=1: bỏ qua — bước này thuộc Version 2.')


full_pipeline -> v3 | {'tag': 'full_checkpoint-750', 'pool': 'top2', 'tau': 0.5, 'lam': 0.2, 'fuse': 'rrf', 'alpha': 0.0, 'wc': 1.0, 'wr': 3.0, 'top': 30, 'm': 8, 'maxlen': 768}


---
## Stage 9V11 — Exact document retrieval + ViLegalBERT auxiliary scorer

Khối này là bản production an toàn hơn của Stage 9:

- gộp document bằng **exact max trên toàn bộ chunk**, không cắt top-1000 chunk trước;
- dùng `ntphuc149/ViLegalBERT` như một semantic scorer phụ ở tầng document/chunk;
- score ViLegalBERT được fuse với reranker AITeamVN đã fine-tune bằng grid trên một
  phần validation và kiểm tra lại trên holdout còn lại;
- nếu ViLegalBERT làm điểm holdout giảm, pipeline tự fallback về bản cũ;
- exact-match memory chỉ áp dụng cho private sau inference, không làm đẹp validation.

`ViLegalBERT` là encoder-only 135M, max 256 token, nên cell dùng tokenizer PhoBERT
và VnCoreNLP word segmentation theo model card. Model này phải được BTC phê duyệt
và tài khoản Hugging Face phải được chấp nhận quyền truy cập model gated.


In [ ]:
#@title Stage 9V11 — exact doc max + ViLegalBERT + safe fusion
import os, re, gc, glob, time, json, random, subprocess, sys, unicodedata
import numpy as np
from collections import defaultdict

# -----------------------------------------------------------------------------
# Exact-match memory: dùng sau cùng cho private, không dùng để chọn config val.
# -----------------------------------------------------------------------------
def _v11_norm_q(text):
    text = unicodedata.normalize('NFC', str(text)).casefold()
    return re.sub(r'\s+', ' ', text).strip()

_v11_memory_sets = defaultdict(set)
for _qid, _row in train_raw.items():
    _ans = tuple(sorted({str(x) for x in (_row.get('answer') or [])}))
    if 0 < len(_ans) <= TOPK_SUBMIT:
        _v11_memory_sets[_v11_norm_q(_row.get('question', ''))].add(_ans)
V11_MEMORY = {
    _q: next(iter(_answers))
    for _q, _answers in _v11_memory_sets.items()
    if len(_answers) == 1
}

def apply_v11_exact_memory(preds, qids):
    out = {q: list(dict.fromkeys(map(str, preds.get(q, [])))) for q in qids}
    hit = 0
    for q in qids:
        known = V11_MEMORY.get(_v11_norm_q(PRIVATE.get(q, QUESTIONS.get(q, ''))))
        if not known:
            continue
        hit += 1
        current = out.get(q, [])
        out[q] = list(known) + [d for d in current if d not in known]
        out[q] = out[q][:TOPK_SUBMIT]
    print(f'V11 exact-memory hit: {hit}/{len(list(qids))}')
    return out

V11_DEPLOY = False
V11_READY = False
V11_FAILURE = None

if RUN_PHASE != 2:
    print('RUN_PHASE=1: V11 chỉ chạy ở Version 2 sau khi đã có fine-tuned reranker.')
else:
    # -------------------------------------------------------------------------
    # V11 config. Không đổi model embedding chính; ViLegalBERT chỉ là nhánh phụ.
    # -------------------------------------------------------------------------
    VILEGAL_MODEL = 'ntphuc149/ViLegalBERT'
    VILEGAL_TOKENIZER = 'vinai/phobert-base-v2'
    VILEGAL_MAXLEN = 256
    V11_CAND_DOCS = 100
    V11_RERANK_TOP = 40
    V11_RERANK_M = 8
    V11_RERANK_MAXLEN = 768
    V11_VILEGAL_M = 2
    V11_VILEGAL_LAM = 0.20
    V11_TUNE_N = min(300, len(VAL_QIDS))
    V11_TUNE_QIDS = list(VAL_QIDS[:V11_TUNE_N])
    V11_HOLD_QIDS = list(VAL_QIDS[V11_TUNE_N:])

    # Exact doc-level max for the existing dense encoder. The old route first
    # selected top-1000 chunks and only then aggregated to documents; that can
    # miss a relevant chunk in a long document.
    if EMB_T is None or qmodel is None:
        load_query_model()
    _v11_chunk_doc_ix = torch.as_tensor(CHUNK2DOC, device=DEV, dtype=torch.long)

    def _v11_chunk_to_doc_max(scores):
        if hasattr(scores, 'scatter_reduce_'):
            out = torch.full(
                (scores.shape[0], len(DOC_IDS)),
                -torch.inf,
                device=scores.device,
                dtype=scores.dtype,
            )
            idx = _v11_chunk_doc_ix.unsqueeze(0).expand(scores.shape[0], -1)
            out.scatter_reduce_(1, idx, scores, reduce='amax', include_self=True)
            return out
        # Compatibility fallback for older torch builds.
        arr = scores.float().detach().cpu().numpy()
        out = np.full((arr.shape[0], len(DOC_IDS)), -np.inf, dtype=np.float32)
        for _r in range(arr.shape[0]):
            np.maximum.at(out[_r], CHUNK2DOC, arr[_r])
        return torch.as_tensor(out, device=scores.device)

    def v11_dense_exact(qids, qtext, topk=V11_CAND_DOCS, batch=16):
        out = {}
        for _i in range(0, len(qtext), batch):
            _qb = qtext[_i:_i + batch]
            _q = qmodel.encode(
                format_dense_queries(_qb),
                batch_size=batch,
                normalize_embeddings=True,
                convert_to_tensor=True,
                show_progress_bar=False,
            ).to(EMB_T.dtype)
            _doc_scores = _v11_chunk_to_doc_max(_q @ EMB_T.T)
            _k = min(topk, _doc_scores.shape[1])
            _, _ix = torch.topk(_doc_scores, _k, dim=1)
            for _qid, _row in zip(qids[_i:_i + batch], _ix.detach().cpu().numpy()):
                out[_qid] = [DOC_IDS[int(_j)] for _j in _row]
        return out

    def _v11_recall_at(preds, gold, k=30):
        vals = []
        for _q, _g in gold.items():
            vals.append(len(set(preds.get(_q, [])[:k]) & set(_g)) / max(1, len(_g)))
        return float(np.mean(vals)) if vals else 0.0

    def _v11_make_candidates(dense_preds, bm25_preds, mode):
        if mode == 'exact_dense':
            return dense_preds
        if mode == 'exact_dense_bm25':
            return rrf(
                dense_preds, bm25_preds,
                k=RRF_K,
                topn=V11_CAND_DOCS,
                weights=(3.0, 0.5),
            )
        raise ValueError(mode)

    # -------------------------------------------------------------------------
    # Load ViLegalBERT. It is an MLM encoder, so use mean pooled contextual
    # embeddings on the selected evidence chunks, not a fake CrossEncoder head.
    # -------------------------------------------------------------------------
    try:
        import torch.nn.functional as F
        from transformers import AutoModel, AutoTokenizer

        _hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
        _hf_kw = {'token': _hf_token} if _hf_token else {}
        try:
            V11_VILEGAL_TOK = AutoTokenizer.from_pretrained(
                VILEGAL_TOKENIZER, use_fast=False, **_hf_kw)
            V11_VILEGAL_MODEL = AutoModel.from_pretrained(
                VILEGAL_MODEL, **_hf_kw)
        except TypeError:
            # Compatibility with older transformers versions.
            _old_kw = {'use_auth_token': _hf_token} if _hf_token else {}
            V11_VILEGAL_TOK = AutoTokenizer.from_pretrained(
                VILEGAL_TOKENIZER, use_fast=False, **_old_kw)
            V11_VILEGAL_MODEL = AutoModel.from_pretrained(
                VILEGAL_MODEL, **_old_kw)

        V11_VILEGAL_MODEL = V11_VILEGAL_MODEL.to(DEV).eval()
        if DEV.startswith('cuda'):
            V11_VILEGAL_MODEL = V11_VILEGAL_MODEL.half()

        # ViLegalBERT's model card recommends PhoBERT-compatible word
        # segmentation before tokenization. Cache the Java model in ART.
        try:
            import py_vncorenlp
        except ImportError:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-q', 'py_vncorenlp'],
                check=True,
            )
            import py_vncorenlp

        V11_SEG_DIR = f'{ART}/models/vncorenlp'
        os.makedirs(V11_SEG_DIR, exist_ok=True)
        if not glob.glob(f'{V11_SEG_DIR}/*.jar'):
            py_vncorenlp.download_model(save_dir=V11_SEG_DIR)
        V11_SEGMENTER = py_vncorenlp.VnCoreNLP(
            annotators=['wseg'], save_dir=V11_SEG_DIR)

        def _v11_segment(text):
            text = re.sub(r'\s+', ' ', str(text)).strip()
            if not text:
                return ''
            try:
                return V11_SEGMENTER.word_segment(text)
            except Exception:
                # Do not crash the whole submission if Java segmentation fails;
                # raw text remains a valid tokenizer input, only less optimal.
                return text

        @torch.inference_mode()
        def v11_vilegal_encode(texts, batch=64):
            rows = []
            texts = list(texts)
            for _i in range(0, len(texts), batch):
                _seg = [_v11_segment(x) for x in texts[_i:_i + batch]]
                _tok = V11_VILEGAL_TOK(
                    _seg,
                    padding=True,
                    truncation=True,
                    max_length=VILEGAL_MAXLEN,
                    return_tensors='pt',
                )
                _tok = {k: v.to(DEV) for k, v in _tok.items()}
                _h = V11_VILEGAL_MODEL(**_tok).last_hidden_state.float()
                _mask = _tok['attention_mask'].unsqueeze(-1).float()
                _v = (_h * _mask).sum(dim=1) / _mask.sum(dim=1).clamp_min(1.0)
                _v = F.normalize(_v, p=2, dim=1)
                rows.append(_v.cpu().numpy().astype(np.float32))
            return np.concatenate(rows, axis=0) if rows else np.empty((0, 768), np.float32)

        V11_READY = True
        print('ViLegalBERT loaded:', VILEGAL_MODEL,
              '| hidden=', V11_VILEGAL_MODEL.config.hidden_size,
              '| maxlen=', VILEGAL_MAXLEN)
    except Exception as _e:
        V11_FAILURE = repr(_e)
        print('Không load được ViLegalBERT -> V11 fallback về pipeline cũ.')
        print('Nếu model gated, hãy accept model card và cấu hình HF_TOKEN trong Kaggle.')
        print(_e)

    if V11_READY:
        # Use the current dense selector to obtain multiple evidence chunks per
        # document, then score those chunks with the legal encoder. This is much
        # cheaper and safer than pretending an MLM checkpoint is a cross-encoder.
        def v11_vilegal_doc_scores(qids, qtext, cand, top=V11_RERANK_TOP,
                                   m=V11_VILEGAL_M):
            _sel_cand = {q: cand[q][:top] for q in qids}
            _selected = select_chunks(qids, qtext, _sel_cand, m=m)
            _qemb = v11_vilegal_encode(qtext, batch=64)
            _chunk_ix = sorted({
                int(ci)
                for q in qids
                for _d, _cs in _selected[q].items()
                for ci, _s in _cs
            })
            _cemb = v11_vilegal_encode(
                [CHUNK_TEXTS[_ci] for _ci in _chunk_ix], batch=64)
            _cpos = {ci: i for i, ci in enumerate(_chunk_ix)}
            _scores = {}
            for _qi, _q in enumerate(qids):
                _scores[_q] = {}
                for _d, _cs in _selected[_q].items():
                    _vals = sorted(
                        [float(_qemb[_qi] @ _cemb[_cpos[int(_ci)]])
                         for _ci, _s in _cs],
                        reverse=True,
                    )
                    if _vals:
                        _scores[_q][_d] = _vals[0] + V11_VILEGAL_LAM * (
                            _vals[1] if len(_vals) > 1 else 0.0)
            return _scores

        # Exact dense candidates + weak BM25 fallback. Select the candidate
        # strategy on the first 300 validation queries, not on private.
        print('V11 exact dense retrieval on validation...', flush=True)
        _v11_dense_val = v11_dense_exact(VAL_QIDS, val_text, V11_CAND_DOCS)
        print('V11 BM25 retrieval on validation...', flush=True)
        _v11_bm25_val = retrieve_bm25(VAL_QIDS, val_text)
        _v11_candidate_options = {
            'exact_dense': _v11_make_candidates(
                _v11_dense_val, _v11_bm25_val, 'exact_dense'),
            'exact_dense_bm25': _v11_make_candidates(
                _v11_dense_val, _v11_bm25_val, 'exact_dense_bm25'),
        }
        _v11_gold = {q: GOLD[q] for q in VAL_QIDS}
        for _name, _p in _v11_candidate_options.items():
            print(
                f'[{_name}] candidate recall@30=',
                f'{_v11_recall_at(_p, _v11_gold, 30):.4f}',
                '| @50=', f'{_v11_recall_at(_p, _v11_gold, 50):.4f}',
            )
        V11_CAND_MODE = max(
            _v11_candidate_options,
            key=lambda _n: (
                _v11_recall_at(
                    {q: _v11_candidate_options[_n][q] for q in V11_TUNE_QIDS},
                    {q: GOLD[q] for q in V11_TUNE_QIDS}, 30),
                _v11_recall_at(
                    {q: _v11_candidate_options[_n][q] for q in V11_TUNE_QIDS},
                    {q: GOLD[q] for q in V11_TUNE_QIDS}, 50),
            ),
        )
        V11_BASE_CAND = _v11_candidate_options[V11_CAND_MODE]
        V11_DENSE_BRANCH = _v11_dense_val
        print('V11 candidate mode ->', V11_CAND_MODE)

        # Reload the fine-tuned AITeamVN reranker in fp32 for inference. The
        # previous run showed score saturation in fp16; this only affects the
        # auxiliary V11 measurement and the final V11 route.
        V11_OLD_FP16 = RERANK_FP16
        RERANK_FP16 = False
        RERANKERS.clear()
        _RR_SCORE.clear()
        gc.collect()
        if torch.cuda.is_available():
            for _gi in range(torch.cuda.device_count()):
                with torch.cuda.device(_gi):
                    torch.cuda.empty_cache()
        for _d in RR_DEVICES:
            get_reranker(_d)

        V11_TAG = 'v11_current_reranker_exact_candidates'
        rerank_collect(
            VAL_QIDS, val_text, V11_BASE_CAND,
            V11_RERANK_TOP, V11_RERANK_M, V11_RERANK_MAXLEN,
            tag=V11_TAG,
        )
        print('V11 ViLegalBERT document scores...', flush=True)
        V11_VILEGAL_SCORES = v11_vilegal_doc_scores(
            VAL_QIDS, val_text, V11_BASE_CAND,
            top=V11_RERANK_TOP, m=V11_VILEGAL_M,
        )

        def _v11_zmap(d):
            vals = np.asarray(list(d.values()), dtype=np.float64)
            if len(vals) == 0:
                return {}
            return {
                k: (float(v) - float(vals.mean())) / (float(vals.std()) + 1e-9)
                for k, v in d.items()
            }

        def _v11_branch_feature(ranking, q, docs):
            pos = {d: i for i, d in enumerate(ranking.get(q, []))}
            return {
                d: (1.0 / (RRF_K + pos[d] + 1) if d in pos else 0.0)
                for d in docs
            }

        def v11_rank(tag, cand, vilegal_scores, dense_branch,
                     qids=None, pool='top2', tau=0.5, lam=0.2,
                     w_vilegal=0.25, w_dense=0.10, w_base=0.05):
            qset = list(qids) if qids is not None else list(cand)
            out = {}
            for _q in qset:
                _scored = [d for d in cand[_q] if d in RRC[tag].get(_q, {})]
                if not _scored:
                    out[_q] = list(cand[_q])
                    continue
                _rr = {
                    d: pool_scores(RRC[tag][_q][d], pool, tau, lam)
                    for d in _scored
                }
                _zrr = _v11_zmap(_rr)
                _v = {d: vilegal_scores.get(_q, {}).get(d, 0.0)
                      for d in _scored}
                _zv = _v11_zmap(_v)
                _dense = _v11_branch_feature(dense_branch, _q, _scored)
                _base = _v11_branch_feature(cand, _q, _scored)
                _zd = _v11_zmap(_dense)
                _zb = _v11_zmap(_base)
                _final = {
                    d: _zrr.get(d, 0.0)
                       + w_vilegal * _zv.get(d, 0.0)
                       + w_dense * _zd.get(d, 0.0)
                       + w_base * _zb.get(d, 0.0)
                    for d in _scored
                }
                out[_q] = [
                    d for d, _s in sorted(_final.items(), key=lambda x: -x[1])
                ] + [d for d in cand[_q] if d not in _final]
            return out

        def _v11_metrics_for(qids, preds):
            _qs = list(qids)
            return evaluate(
                {q: preds[q] for q in _qs},
                {q: GOLD[q] for q in _qs},
            )

        # Small CPU-only fusion grid. Selection is on 300 queries; the remaining
        # validation queries are used as a gate before private deployment.
        _v11_grid = []
        for _pool in ('max', 'top2', 'lse'):
            for _lam in (0.10, 0.20, 0.30):
                for _wv in (0.0, 0.10, 0.20, 0.40, 0.60):
                    for _wd in (0.0, 0.05, 0.10, 0.20):
                        _pred = v11_rank(
                            V11_TAG, V11_BASE_CAND, V11_VILEGAL_SCORES,
                            V11_DENSE_BRANCH, qids=V11_TUNE_QIDS,
                            pool=_pool, lam=_lam, w_vilegal=_wv, w_dense=_wd,
                        )
                        _m = _v11_metrics_for(V11_TUNE_QIDS, _pred)
                        _v11_grid.append({
                            'pool': _pool, 'lam': _lam,
                            'w_vilegal': _wv, 'w_dense': _wd,
                            'w_base': 0.05,
                            'OFFICIAL_recall': _m['OFFICIAL_recall'],
                            'OFFICIAL_precision': _m['OFFICIAL_precision'],
                        })
        _v11_grid = sorted(
            _v11_grid,
            key=lambda x: (x['OFFICIAL_recall'], x['OFFICIAL_precision']),
            reverse=True,
        )
        V11_CONFIG = _v11_grid[0]
        print('V11 best tune config:', V11_CONFIG)

        V11_VAL_PREDS = v11_rank(
            V11_TAG, V11_BASE_CAND, V11_VILEGAL_SCORES,
            V11_DENSE_BRANCH, pool=V11_CONFIG['pool'],
            lam=V11_CONFIG['lam'], w_vilegal=V11_CONFIG['w_vilegal'],
            w_dense=V11_CONFIG['w_dense'], w_base=V11_CONFIG['w_base'],
        )
        V11_VAL_METRIC = _v11_metrics_for(VAL_QIDS, V11_VAL_PREDS)

        if 'BEST' in globals():
            _old_best_preds = rank_from(
                BEST['tag'], BASES[RERANK_BASE], BEST['pool'], BEST['tau'],
                BEST['lam'], BEST['fuse'], BEST['alpha'], BEST['wc'], BEST['wr'],
            )
        else:
            _old_best_preds = dense_val

        _old_hold = _v11_metrics_for(
            V11_HOLD_QIDS,
            {q: _old_best_preds[q] for q in V11_HOLD_QIDS},
        )
        _new_hold = _v11_metrics_for(
            V11_HOLD_QIDS,
            {q: V11_VAL_PREDS[q] for q in V11_HOLD_QIDS},
        )
        print('[old best] holdout:', _old_hold)
        print('[V11] full val:', V11_VAL_METRIC)
        print('[V11] holdout:', _new_hold)
        V11_DEPLOY = (
            (_new_hold['OFFICIAL_recall'], _new_hold['OFFICIAL_precision'])
            >= (_old_hold['OFFICIAL_recall'], _old_hold['OFFICIAL_precision'])
        )
        print('V11 deploy gate ->', V11_DEPLOY,
              '| ViLegalBERT is used only when holdout does not regress.')

        def full_pipeline_v11(qids, qtext):
            qids = list(qids)
            _dense = v11_dense_exact(qids, qtext, V11_CAND_DOCS)
            _bm = retrieve_bm25(qids, qtext)
            _cand = _v11_make_candidates(_dense, _bm, V11_CAND_MODE)
            _tag = 'private_v11_rerank'
            rerank_collect(
                qids, qtext, _cand,
                V11_RERANK_TOP, V11_RERANK_M, V11_RERANK_MAXLEN,
                tag=_tag,
            )
            _vl = v11_vilegal_doc_scores(
                qids, qtext, _cand,
                top=V11_RERANK_TOP, m=V11_VILEGAL_M,
            )
            return v11_rank(
                _tag, _cand, _vl, _dense,
                pool=V11_CONFIG['pool'], tau=0.5,
                lam=V11_CONFIG['lam'],
                w_vilegal=V11_CONFIG['w_vilegal'],
                w_dense=V11_CONFIG['w_dense'],
                w_base=V11_CONFIG['w_base'],
            )

        if V11_DEPLOY:
            full_pipeline = full_pipeline_v11
            print('full_pipeline -> V11 exact dense + ViLegalBERT + current reranker')
        else:
            print('full_pipeline -> v3b baseline vì V11 không thắng holdout')

        # Close the Java segmenter only after private inference; keep the model
        # and cached functions alive for full_pipeline_v11.
        V11_OLD_FP16 = RERANK_FP16

---
## Stage 9V12 — Fine-tuned ViLegalBERT legal cross-encoder

V11 dùng ViLegalBERT như embedding phụ. V12 fine-tune thêm một classification head
trên chính query/gold/hard-negative của Task 1:

- backbone: `ntphuc149/ViLegalBERT`;
- input: cặp `(question, legal window)` đã word-segment bằng VnCoreNLP;
- window: 224 token, overlap 32 token, tối đa 3 window/chunk;
- loss: binary ranking với hard negatives từ dense retrieval;
- production: fuse score V12 với AITeamVN reranker, dense rank và citation/title feature;
- holdout gate tự động fallback nếu V12 không thắng.

Chạy `RUN_PHASE=1` để train và lưu checkpoint; sau đó Save Version rồi chạy
`RUN_PHASE=2` để benchmark và dự đoán private.


In [ ]:
#@title Stage 9V12 — fine-tune ViLegalBERT cross-encoder + citation fusion
import os, re, gc, glob, time, json, random, subprocess, sys, unicodedata
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader

V12_ENABLED = True
V12_CE_MODEL = 'ntphuc149/ViLegalBERT'
V12_CE_TOKENIZER = 'vinai/phobert-base-v2'
V12_CE_MAXLEN = 256
V12_WINDOW_TOKENS = 224
V12_WINDOW_OVERLAP = 32
V12_WINDOWS_PER_CHUNK = 3
V12_TRAIN_NEGS = 11
V12_TRAIN_POS = 2
V12_TRAIN_EPOCHS = 1
V12_TRAIN_LR = 2e-5
V12_TRAIN_BATCH = 8
V12_TRAIN_ACCUM = 2
V12_CE_REL = 'models/vilegalbert-crossencoder-v12'
V12_CE_LOCAL = f'{ART}/{V12_CE_REL}'
V12_USE_ALL_TRAIN = False  # bật True chỉ sau khi đã chốt hyperparameter trên holdout
V12_DEPLOY = False
V12_READY = False
V12_TRAINED = False
V12_FAILURE = None

def _v12_find_checkpoint():
    if os.path.isdir(V12_CE_LOCAL) and os.path.exists(f'{V12_CE_LOCAL}/model.pt'):
        return V12_CE_LOCAL
    for _d in ART_IN:
        _p = f'{_d}/{V12_CE_REL}'
        if os.path.exists(f'{_p}/model.pt'):
            return _p
    return None

try:
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

    _v12_hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    _v12_hf_kw = {'token': _v12_hf_token} if _v12_hf_token else {}
    try:
        V12_TOK = AutoTokenizer.from_pretrained(
            V12_CE_TOKENIZER, use_fast=False, **_v12_hf_kw)
    except TypeError:
        V12_TOK = AutoTokenizer.from_pretrained(
            V12_CE_TOKENIZER, use_fast=False,
            **({'use_auth_token': _v12_hf_token} if _v12_hf_token else {}))

    try:
        import py_vncorenlp
    except ImportError:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', 'py_vncorenlp'],
            check=True,
        )
        import py_vncorenlp

    V12_SEG_DIR = f'{ART}/models/vncorenlp'
    os.makedirs(V12_SEG_DIR, exist_ok=True)
    if not glob.glob(f'{V12_SEG_DIR}/*.jar'):
        py_vncorenlp.download_model(save_dir=V12_SEG_DIR)
    V12_SEGMENTER = py_vncorenlp.VnCoreNLP(
        annotators=['wseg'], save_dir=V12_SEG_DIR)

    from functools import lru_cache

    @lru_cache(maxsize=250000)
    def v12_segment(text):
        text = re.sub(r'\s+', ' ', str(text)).strip()
        if not text:
            return ''
        try:
            return V12_SEGMENTER.word_segment(text)
        except Exception:
            return text

    class V12ViLegalCrossEncoder(nn.Module):
        def __init__(self, model_name, token_kw=None):
            super().__init__()
            token_kw = token_kw or {}
            try:
                self.backbone = AutoModel.from_pretrained(model_name, **token_kw)
            except TypeError:
                self.backbone = AutoModel.from_pretrained(
                    model_name,
                    **({'use_auth_token': _v12_hf_token} if _v12_hf_token else {}),
                )
            self.dropout = nn.Dropout(0.10)
            self.classifier = nn.Linear(self.backbone.config.hidden_size, 1)

        def forward(self, input_ids, attention_mask, **kwargs):
            # PhoBERT does not need token_type_ids; passing only the two stable
            # inputs also keeps this compatible with different transformers versions.
            out = self.backbone(
                input_ids=input_ids,
                attention_mask=attention_mask,
            ).last_hidden_state
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (out.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
            return self.classifier(self.dropout(pooled)).squeeze(-1)

    def v12_tokenize_pairs(queries, docs):
        return V12_TOK(
            [v12_segment(q) for q in queries],
            [v12_segment(d) for d in docs],
            padding=True,
            truncation=True,
            max_length=V12_CE_MAXLEN,
            return_tensors='pt',
        )

    class V12PairDataset(TorchDataset):
        def __init__(self, rows):
            self.rows = rows
        def __len__(self):
            return len(self.rows)
        def __getitem__(self, i):
            return self.rows[i]

    def v12_collate(batch):
        qs = [x[0] for x in batch]
        ds = [x[1] for x in batch]
        ys = torch.tensor([x[2] for x in batch], dtype=torch.float32)
        tok = v12_tokenize_pairs(qs, ds)
        return tok, ys

    def v12_build_train_rows(qids):
        qids = list(qids)
        qtext = [QUESTIONS[q] for q in qids]
        print('V12 mine dense candidates...', flush=True)
        cand = retrieve_dense(qids, qtext)
        pos_rep = select_chunks(
            qids, qtext, {q: list(GOLD[q]) for q in qids}, m=3)
        neg_rep = select_chunks(
            qids, qtext, {q: cand[q][:40] for q in qids}, m=1)
        rng = random.Random(SEED + 1200)
        rows, skipped = [], 0
        for q in qids:
            gold = GOLD[q]
            pos_chunks = []
            for g in sorted(gold):
                for ci, _s in pos_rep[q].get(g, [])[:3]:
                    if int(ci) not in pos_chunks:
                        pos_chunks.append(int(ci))
                    if len(pos_chunks) >= V12_TRAIN_POS:
                        break
                if len(pos_chunks) >= V12_TRAIN_POS:
                    break
            neg_docs = [d for d in cand[q][:40]
                        if d not in gold and d in neg_rep[q]]
            if not pos_chunks or len(neg_docs) < 2:
                skipped += 1
                continue
            neg_docs = rng.sample(neg_docs, min(V12_TRAIN_NEGS, len(neg_docs)))
            for ci in pos_chunks:
                rows.append((QUESTIONS[q], CHUNK_TEXTS[ci], 1.0))
            for d in neg_docs:
                rows.append((QUESTIONS[q], CHUNK_TEXTS[neg_rep[q][d][0][0]], 0.0))
        print(f'V12 rows={len(rows)} | skipped queries={skipped}', flush=True)
        if not rows:
            raise RuntimeError('V12 không sinh được training rows')
        return rows

    def v12_save_model(model, path):
        os.makedirs(path, exist_ok=True)
        torch.save({'state_dict': model.state_dict()}, f'{path}/model.pt')
        with open(f'{path}/config.json', 'w', encoding='utf-8') as f:
            json.dump({
                'backbone': V12_CE_MODEL,
                'tokenizer': V12_CE_TOKENIZER,
                'max_length': V12_CE_MAXLEN,
                'window_tokens': V12_WINDOW_TOKENS,
            }, f, ensure_ascii=False, indent=2)

    # Train only in Phase 1. The selected validation split remains untouched.
    if RUN_PHASE == 1:
        # Stage 8 deliberately released the dense encoder before reranker
        # training. Reload it only for hard-negative mining, then release it
        # again before allocating ViLegalBERT so the extra model fits in VRAM.
        if EMB_T is None or qmodel is None:
            load_query_model()
        _v12_train_qids = list(train_raw.keys()) if V12_USE_ALL_TRAIN else list(TRAIN_QIDS)
        _v12_rows = v12_build_train_rows(_v12_train_qids)
        free_dense()
        if 'RERANKERS' in globals():
            RERANKERS.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _v12_model = V12ViLegalCrossEncoder(V12_CE_MODEL, _v12_hf_kw)
        _v12_model = _v12_model.to(DEV)
        if hasattr(_v12_model.backbone, 'gradient_checkpointing_enable'):
            try:
                _v12_model.backbone.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs={'use_reentrant': False})
            except TypeError:
                _v12_model.backbone.gradient_checkpointing_enable()
        # Freeze the word embedding table; the legal continual-pretraining is
        # already in the backbone, while the transformer layers and head adapt
        # to Task 1 ranking.
        for _p in _v12_model.backbone.get_input_embeddings().parameters():
            _p.requires_grad_(False)
        _v12_loader = DataLoader(
            V12PairDataset(_v12_rows),
            batch_size=V12_TRAIN_BATCH,
            shuffle=True,
            drop_last=False,
            collate_fn=v12_collate,
        )
        _v12_opt = torch.optim.AdamW(
            [p for p in _v12_model.parameters() if p.requires_grad],
            lr=V12_TRAIN_LR,
            weight_decay=0.01,
        )
        _v12_steps = max(1, int(np.ceil(len(_v12_loader) / V12_TRAIN_ACCUM)) * V12_TRAIN_EPOCHS)
        _v12_sched = get_linear_schedule_with_warmup(
            _v12_opt, num_warmup_steps=max(1, _v12_steps // 10),
            num_training_steps=_v12_steps)
        _v12_pos_weight = torch.tensor(
            max(1.0, sum(x[2] == 0 for x in _v12_rows) /
                max(1, sum(x[2] == 1 for x in _v12_rows))),
            device=DEV,
        )
        _v12_loss_fn = nn.BCEWithLogitsLoss(pos_weight=_v12_pos_weight)
        _v12_scaler = torch.cuda.amp.GradScaler(enabled=DEV.startswith('cuda'))
        _v12_model.train()
        _v12_opt.zero_grad(set_to_none=True)
        _v12_step = 0
        _v12_t0 = time.time()
        for _ep in range(V12_TRAIN_EPOCHS):
            _v12_running = 0.0
            for _bi, (_tok, _y) in enumerate(_v12_loader):
                _tok = {k: v.to(DEV) for k, v in _tok.items()}
                _y = _y.to(DEV)
                with torch.cuda.amp.autocast(enabled=DEV.startswith('cuda')):
                    _logit = _v12_model(**_tok)
                    _loss = _v12_loss_fn(_logit.float(), _y) / V12_TRAIN_ACCUM
                _v12_scaler.scale(_loss).backward()
                _v12_running += float(_loss.detach().cpu())
                if ((_bi + 1) % V12_TRAIN_ACCUM == 0) or (_bi + 1 == len(_v12_loader)):
                    _v12_scaler.unscale_(_v12_opt)
                    torch.nn.utils.clip_grad_norm_(_v12_model.parameters(), 1.0)
                    _v12_scaler.step(_v12_opt)
                    _v12_scaler.update()
                    _v12_sched.step()
                    _v12_opt.zero_grad(set_to_none=True)
                    _v12_step += 1
                if (_bi + 1) % 200 == 0:
                    print(f'V12 epoch={_ep+1} batch={_bi+1}/{len(_v12_loader)} '
                          f'loss={_v12_running/(200 if _bi >= 199 else _bi+1):.4f}', flush=True)
                    _v12_running = 0.0
        _v12_model.eval()
        v12_save_model(_v12_model, V12_CE_LOCAL)
        V12_TRAINED = True
        V12_READY = True
        print('V12 train done: %.1f min -> %s' %
              ((time.time() - _v12_t0) / 60, V12_CE_LOCAL))
        del _v12_model, _v12_loader, _v12_opt, _v12_sched, _v12_rows
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        _v12_ckpt = _v12_find_checkpoint()
        if _v12_ckpt is None:
            print('Không thấy checkpoint V12 -> giữ pipeline V11/cũ. '
                  'Chạy RUN_PHASE=1 trước để train ViLegalBERT.')
        else:
            _v12_model = V12ViLegalCrossEncoder(V12_CE_MODEL, _v12_hf_kw)
            _v12_state = torch.load(f'{_v12_ckpt}/model.pt', map_location='cpu')
            _v12_model.load_state_dict(_v12_state['state_dict'], strict=True)
            _v12_model = _v12_model.to(DEV).eval()
            V12_READY = True
            print('V12 checkpoint <-', _v12_ckpt)

    if V12_READY and RUN_PHASE == 2:
        # V11 normally leaves these loaded, but keep V12 self-contained if
        # V11 was skipped or fell back before initializing dense retrieval.
        if EMB_T is None or qmodel is None:
            load_query_model()
        @torch.inference_mode()
        def v12_score_pairs(queries, docs, batch=32):
            out = []
            for _i in range(0, len(queries), batch):
                _tok = v12_tokenize_pairs(queries[_i:_i+batch], docs[_i:_i+batch])
                _tok = {k: v.to(DEV) for k, v in _tok.items()}
                _logit = _v12_model(**_tok).float()
                out.extend(_logit.detach().cpu().numpy().tolist())
            return np.asarray(out, dtype=np.float64)

        def v12_windows(text):
            toks = v12_segment(text).split()
            if len(toks) <= V12_WINDOW_TOKENS:
                return [' '.join(toks)]
            starts = [
                0,
                max(0, (len(toks) - V12_WINDOW_TOKENS) // 2),
                max(0, len(toks) - V12_WINDOW_TOKENS),
            ]
            out, seen = [], set()
            for _s in starts:
                _w = ' '.join(toks[_s:_s + V12_WINDOW_TOKENS])
                if _w and _w not in seen:
                    seen.add(_w); out.append(_w)
                if len(out) >= V12_WINDOWS_PER_CHUNK:
                    break
            return out

        def v12_cross_doc_scores(qids, qtext, cand, top=40, m=2):
            _sel = select_chunks(qids, qtext, {q: cand[q][:top] for q in qids}, m=m)
            _pairs_q, _pairs_d, _owners = [], [], []
            for _q, _qt in zip(qids, qtext):
                for _d, _cs in _sel[_q].items():
                    for _ci, _s in _cs:
                        for _w in v12_windows(CHUNK_TEXTS[int(_ci)]):
                            _pairs_q.append(_qt)
                            _pairs_d.append(_w)
                            _owners.append((_q, _d))
            _scores = v12_score_pairs(_pairs_q, _pairs_d, batch=V12_TRAIN_BATCH * 2)
            _per = defaultdict(list)
            for (_q, _d), _s in zip(_owners, _scores):
                _per[(_q, _d)].append(float(_s))
            out = defaultdict(dict)
            for (_q, _d), _ss in _per.items():
                _ss = sorted(_ss, reverse=True)
                out[_q][_d] = _ss[0] + 0.20 * (_ss[1] if len(_ss) > 1 else 0.0)
            return dict(out)

        # Citation/title feature. It is deliberately a small feature, never a
        # hard override; semantic scores remain dominant.
        _v12_meta = {}
        try:
            for _r in corpus.itertuples(index=False):
                _v12_meta[str(_r.doc_id)] = (
                    str(getattr(_r, 'title', '')) + ' ' +
                    str(getattr(_r, 'link', ''))
                )
        except Exception:
            _v12_meta = {str(d): str(d) for d in DOC_IDS}

        def v12_rule_score(question, doc_id):
            q = unicodedata.normalize('NFC', str(question)).casefold()
            d = unicodedata.normalize('NFC', _v12_meta.get(str(doc_id), '')).casefold()
            score = 0.0
            # Exact legal-document code: 123/2020/NĐ-CP, 01/2022/TT-BTP, ...
            codes = re.findall(
                r'\b\d{1,4}\s*/\s*\d{4}\s*/\s*[a-zđ]{2,12}(?:-[a-zđ]{1,12})?\b',
                q,
            )
            for code in codes:
                if re.sub(r'\s+', '', code) in re.sub(r'\s+', '', d):
                    score += 2.0
            # Number and legal-type overlap is weaker than exact code overlap.
            qnums = set(re.findall(r'\b\d{1,4}\b', q))
            dnums = set(re.findall(r'\b\d{1,4}\b', d))
            if qnums and qnums & dnums:
                score += min(0.5, 0.15 * len(qnums & dnums))
            for term in ('nghị định', 'thông tư', 'quyết định', 'luật',
                         'nghị quyết', 'bộ luật', 'chỉ thị'):
                if term in q and term in d:
                    score += 0.15
            return score

        def _v12_z(d):
            vals = np.asarray(list(d.values()), dtype=np.float64)
            if len(vals) == 0:
                return {}
            return {k: (float(v) - vals.mean()) / (vals.std() + 1e-9)
                    for k, v in d.items()}

        def _v12_rr_feature(ranking, q, docs):
            pos = {d: i for i, d in enumerate(ranking.get(q, []))}
            return {d: (1.0 / (RRF_K + pos[d] + 1) if d in pos else 0.0)
                    for d in docs}

        def v12_rank(cand, current_scores, ce_scores, dense_branch,
                     qids=None, w_ce=0.70, w_rule=0.10, w_dense=0.10):
            qset = list(qids) if qids is not None else list(cand)
            out = {}
            for _q in qset:
                _docs = list(cand[_q][:40])
                _cur = {d: current_scores.get(_q, {}).get(d, 0.0) for d in _docs}
                _ce = {d: ce_scores.get(_q, {}).get(d, 0.0) for d in _docs}
                _rule = {d: v12_rule_score(QUESTIONS.get(_q, PRIVATE.get(_q, '')), d)
                         for d in _docs}
                _dense = _v12_rr_feature(dense_branch, _q, _docs)
                _zcur, _zce = _v12_z(_cur), _v12_z(_ce)
                _zrule, _zdense = _v12_z(_rule), _v12_z(_dense)
                _score = {
                    d: _zcur.get(d, 0.0)
                       + w_ce * _zce.get(d, 0.0)
                       + w_rule * _zrule.get(d, 0.0)
                       + w_dense * _zdense.get(d, 0.0)
                    for d in _docs
                }
                out[_q] = [d for d, _s in sorted(_score.items(), key=lambda x: -x[1])]
                out[_q] += [d for d in cand[_q] if d not in _score]
            return out

        # Reuse V11 candidates where available; otherwise construct exact dense
        # candidates here. The current AITeamVN reranker is kept as a feature.
        if 'V11_BASE_CAND' in globals() and 'V11_DENSE_BRANCH' in globals():
            V12_BASE_CAND = V11_BASE_CAND
            V12_DENSE_BRANCH = V11_DENSE_BRANCH
        else:
            if 'v11_dense_exact' in globals():
                V12_DENSE_BRANCH = v11_dense_exact(VAL_QIDS, val_text, 100)
            else:
                V12_DENSE_BRANCH = retrieve_dense(VAL_QIDS, val_text)
            _v12_bm = retrieve_bm25(VAL_QIDS, val_text)
            V12_BASE_CAND = (
                rrf(V12_DENSE_BRANCH, _v12_bm, k=RRF_K, topn=100,
                    weights=(3.0, 0.5))
            )

        # Current reranker scores. Reuse V11's cached scores when possible.
        V12_CURRENT_TAG = globals().get('V11_TAG', None)
        if V12_CURRENT_TAG not in RRC:
            V12_CURRENT_TAG = 'v12_current_reranker'
            rerank_collect(
                VAL_QIDS, val_text, V12_BASE_CAND,
                40, 8, 768, tag=V12_CURRENT_TAG)
        V12_CURRENT_SCORES = {
            q: {d: pool_scores(v, 'top2', lam=0.20)
                for d, v in RRC[V12_CURRENT_TAG].get(q, {}).items()}
            for q in VAL_QIDS
        }
        print('V12 fine-tuned cross-encoder scores on validation...', flush=True)
        V12_CE_SCORES = v12_cross_doc_scores(
            VAL_QIDS, val_text, V12_BASE_CAND, top=40, m=2)

        # Tune only fusion weights on 300 queries; use the rest as a gate.
        _v12_grid = []
        for _wce in (0.30, 0.50, 0.70, 0.90, 1.10):
            for _wrule in (0.0, 0.05, 0.10, 0.20, 0.30):
                for _wdense in (0.0, 0.05, 0.10, 0.20):
                    _p = v12_rank(
                        V12_BASE_CAND, V12_CURRENT_SCORES, V12_CE_SCORES,
                        V12_DENSE_BRANCH, qids=V11_TUNE_QIDS,
                        w_ce=_wce, w_rule=_wrule, w_dense=_wdense)
                    _m = evaluate(
                        _p, {q: GOLD[q] for q in V11_TUNE_QIDS})
                    _v12_grid.append({
                        'w_ce': _wce, 'w_rule': _wrule, 'w_dense': _wdense,
                        'OFFICIAL_recall': _m['OFFICIAL_recall'],
                        'OFFICIAL_precision': _m['OFFICIAL_precision'],
                    })
        V12_CONFIG = max(
            _v12_grid,
            key=lambda x: (x['OFFICIAL_recall'], x['OFFICIAL_precision']))
        V12_VAL_PREDS = v12_rank(
            V12_BASE_CAND, V12_CURRENT_SCORES, V12_CE_SCORES,
            V12_DENSE_BRANCH, w_ce=V12_CONFIG['w_ce'],
            w_rule=V12_CONFIG['w_rule'], w_dense=V12_CONFIG['w_dense'])

        # Gate V12 against the best route already selected by V11/baseline.
        if 'V11_VAL_PREDS' in globals():
            _v12_old = V11_VAL_PREDS
        elif 'best_preds' in globals():
            _v12_old = best_preds
        else:
            _v12_old = dense_val
        _v12_hold = set(V11_HOLD_QIDS) if 'V11_HOLD_QIDS' in globals() else set(VAL_QIDS)
        _old_m = evaluate({q: _v12_old[q] for q in _v12_hold},
                           {q: GOLD[q] for q in _v12_hold})
        _new_m = evaluate({q: V12_VAL_PREDS[q] for q in _v12_hold},
                           {q: GOLD[q] for q in _v12_hold})
        print('V12 config:', V12_CONFIG)
        print('V12 old holdout:', _old_m)
        print('V12 new holdout:', _new_m)
        V12_DEPLOY = (
            (_new_m['OFFICIAL_recall'], _new_m['OFFICIAL_precision'])
            >= (_old_m['OFFICIAL_recall'], _old_m['OFFICIAL_precision'])
        )
        print('V12 deploy gate ->', V12_DEPLOY)

        def full_pipeline_v12(qids, qtext):
            qids = list(qids)
            if 'v11_dense_exact' in globals():
                _dense = v11_dense_exact(qids, qtext, 100)
            else:
                _dense = retrieve_dense(qids, qtext)
            _bm = retrieve_bm25(qids, qtext)
            _cand = rrf(_dense, _bm, k=RRF_K, topn=100,
                        weights=(3.0, 0.5))
            _tag = 'private_v12_current_reranker'
            rerank_collect(qids, qtext, _cand, 40, 8, 768, tag=_tag)
            _current = {
                q: {d: pool_scores(v, 'top2', lam=0.20)
                    for d, v in RRC[_tag].get(q, {}).items()}
                for q in qids
            }
            _ce = v12_cross_doc_scores(qids, qtext, _cand, top=40, m=2)
            return v12_rank(
                _cand, _current, _ce, _dense, qids=qids,
                w_ce=V12_CONFIG['w_ce'],
                w_rule=V12_CONFIG['w_rule'],
                w_dense=V12_CONFIG['w_dense'],
            )

        if V12_DEPLOY:
            full_pipeline = full_pipeline_v12
            print('full_pipeline -> V12 fine-tuned ViLegalBERT CE + AITeamVN')
        else:
            print('V12 không thắng holdout -> giữ full_pipeline hiện tại')
except Exception as _e:
    V12_FAILURE = repr(_e)
    print('V12 disabled:', _e)
    print('Pipeline trước đó vẫn được giữ nguyên.')

In [27]:
#@title Chạy pipeline trên private test
import time

if RUN_PHASE == 2:
    assert 'BEST' in globals(), 'Chưa có BEST — chạy cell 7B.4b trước.'
    print(f"pipeline v3: base='{RERANK_BASE}' | {BEST}")
    private_qids = list(PRIVATE.keys())
    private_text = [PRIVATE[q] for q in private_qids]
    t0 = time.time()
    private_preds = full_pipeline(private_qids, private_text)
    if 'apply_v11_exact_memory' in globals():
        private_preds = apply_v11_exact_memory(private_preds, private_qids)
    print('%d câu | %.1f phút' % (len(private_qids), (time.time() - t0) / 60))
else:
    print('RUN_PHASE=1: bỏ private inference; bước này thuộc Version 2.')


pipeline v3: base='dense' | {'tag': 'full_checkpoint-750', 'pool': 'top2', 'tau': 0.5, 'lam': 0.2, 'fuse': 'rrf', 'alpha': 0.0, 'wc': 1.0, 'wr': 3.0, 'top': 30, 'm': 8, 'maxlen': 768}
[private_full_checkpoint-750] chọn chunk 4s | 7.56 chunk/doc thực nhận
  200/2080 query
  400/2080 query
  600/2080 query
  800/2080 query
  1000/2080 query
  1200/2080 query
  1400/2080 query
  1600/2080 query
  1800/2080 query
  2000/2080 query
[private_full_checkpoint-750] 471584 cặp | 108.3 phút
[private_full_checkpoint-750] điểm: min=-11.430 p50=-11.086 max=11.391 std=2.653 | phân biệt 1.9%
2080 câu | 108.4 phút


In [28]:
if RUN_PHASE == 2:
    #@title Validate + zip
    import json, zipfile, os, collections

    def build_submission(preds, all_qids, corpus_ids):
        """Luôn trả về ĐÚNG 5 id/câu. Thiếu thì bù bằng ứng viên tốt nhất còn lại."""
        sub, filler = {}, [d for d in DOC_IDS[:TOPK_SUBMIT]]
        for q in all_qids:
            ids, seen = [], set()
            for d in preds.get(q, []):
                d = str(d)
                if d not in seen and d in corpus_ids:
                    seen.add(d); ids.append(d)
                if len(ids) == TOPK_SUBMIT:
                    break
            for f in filler:                       # câu không có ứng viên -> vẫn đủ 5
                if len(ids) == TOPK_SUBMIT:
                    break
                if f not in seen:
                    seen.add(f); ids.append(f)
            sub[q] = {'answer': ids}
        return sub

    def validate(sub, all_qids, corpus_ids):
        errs = []
        # scoring.py của BTC: len(ids_preds) != len(ids_truth) -> raise Exception
        # -> lệch số câu làm SẬP TOÀN BỘ bài nộp, không phải chỉ mất điểm 1 câu.
        if len(sub) != len(all_qids):
            errs.append(f'SẬP TOÀN BÀI: {len(sub)} câu nộp vs {len(all_qids)} câu đề')
        if set(sub) != set(all_qids):
            errs.append(f'SẬP TOÀN BÀI: thiếu {len(set(all_qids)-set(sub))} qid, '
                        f'thừa {len(set(sub)-set(all_qids))} qid (len(None) -> TypeError)')
        for q, v in sub.items():
            a = v.get('answer')
            if not isinstance(a, list):
                errs.append(f'{q}: answer không phải list'); continue
            if len(a) == 0:
                errs.append(f'{q}: answer rỗng -> câu này 0 điểm')
            if len(a) > TOPK_SUBMIT:
                errs.append(f'{q}: {len(a)} id > {TOPK_SUBMIT} -> CÂU NÀY BỊ 0 ĐIỂM')
            if len(a) < TOPK_SUBMIT:
                errs.append(f'{q}: chỉ {len(a)} id -> mất điểm Recall vô cớ')
            if len(set(a)) != len(a):
                errs.append(f'{q}: có id trùng lặp')
            if any(not isinstance(x, str) for x in a):
                errs.append(f'{q}: id phải là string')
            if any(x not in corpus_ids for x in a):
                errs.append(f'{q}: có id không tồn tại trong corpus')
        return errs

    corpus_ids = set(corpus.doc_id)
    sub = build_submission(private_preds, list(PRIVATE), corpus_ids)
    errs = validate(sub, list(PRIVATE), corpus_ids)

    if errs:
        print(f'{len(errs)} LỖI — chưa nộp được:')
        for e in errs[:20]:
            print('  -', e)
    else:
        # Kaggle: ghi thẳng vào GỐC /kaggle/working -> hiện ngay tab Output, tải 1 click.
        # KHÔNG ghi vào /kaggle/temp: thư mục đó không vào output, session chết là mất file.
        SUB_DIR  = '/kaggle/working' if ON_KAGGLE else ART
        out_json = f'{SUB_DIR}/submission.json'
        out_zip  = f'{SUB_DIR}/submission_{PROFILE}.zip'
        with open(out_json, 'w', encoding='utf-8') as f:
            json.dump(sub, f, ensure_ascii=False)
        with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
            z.write(out_json, arcname='submission.json')   # arcname -> nằm ở GỐC zip
        with zipfile.ZipFile(out_zip) as z:
            assert z.namelist() == ['submission.json'], z.namelist()
        print('OK ->', out_zip, '(%.0f KB)' % (os.path.getsize(out_zip) / 1e3))
        print('     ', out_json)
        if ON_KAGGLE:
            print('Tải về: panel bên phải -> Output -> /kaggle/working -> '
                  f'submission_{PROFILE}.zip (hoặc Save Version rồi tải từ tab Output).')
            print('Nộp file ZIP lên Codabench, không nộp file .json.')
        print('phân bố số id/câu:', collections.Counter(len(v['answer']) for v in sub.values()))
        print(json.dumps(dict(list(sub.items())[:2]), ensure_ascii=False, indent=1))
else:
    print('RUN_PHASE=1: bỏ qua — bước này thuộc Version 2.')


OK -> /kaggle/working/submission_v3b_ft_reranker_improved.zip (41 KB)
      /kaggle/working/submission.json
Tải về: panel bên phải -> Output -> /kaggle/working -> submission_v3b_ft_reranker_improved.zip (hoặc Save Version rồi tải từ tab Output).
Nộp file ZIP lên Codabench, không nộp file .json.
phân bố số id/câu: Counter({5: 2080})
{
 "44474": {
  "answer": [
   "200618",
   "292631",
   "222640",
   "26173",
   "155884"
  ]
 },
 "77938": {
  "answer": [
   "136664",
   "52552",
   "146911",
   "298075",
   "226911"
  ]
 }
}


---
## Stage 10 — Xem lại log thí nghiệm

In [29]:
if RUN_PHASE == 2:
    #@title Bảng thí nghiệm
    import pandas as pd, os
    if os.path.exists(LOG_CSV):
        df = pd.read_csv(LOG_CSV)
        cols = [c for c in ['time','profile','run','OFFICIAL_recall',
                            'recall@5','recall@20','recall@100','note'] if c in df.columns]
        display(df[cols].tail(25))
    else:
        print('chưa có log')
else:
    print('RUN_PHASE=1: bỏ qua — bước này thuộc Version 2.')


,time,profile,run,OFFICIAL_recall,recall@5,recall@20,recall@100,note
0,2026-09-20T15:15:50,v3b_ft_reranker_improved,bm25,0.691750,0.691750,0.859667,0.948417,chunk=3000 k1=0.9 b=0.4
1,2026-09-20T15:16:22,v3b_ft_reranker_improved,dense_ft,0.928333,0.928333,0.979417,0.987917,/kaggle/input/datasets/kitnehi1211/v3b-finetun...
2,2026-09-20T15:16:22,v3b_ft_reranker_improved,hybrid_rrf_w,0.930750,0.930750,0.980667,0.987917,"K=60 w=(1, 5)"
3,2026-09-20T17:29:12,v3b_ft_reranker_improved,rerank_ft_checkpoint-600,0.951167,0.951167,0.979667,0.987917,/kaggle/input/datasets/kitnehi1211/v3b-finetun...
4,2026-09-20T18:21:08,v3b_ft_reranker_improved,rerank_ft_checkpoint-750,0.952917,0.952917,0.979917,0.987917,/kaggle/input/datasets/kitnehi1211/v3b-finetun...
5,2026-09-20T19:25:29,v3b_ft_reranker_improved,7B_best,0.958500,0.958500,0.980917,0.987917,"{'tag': 'full_checkpoint-750', 'pool': 'top2',..."


---
## Stage 11 — Lưu artifact để session sau dùng lại (Kaggle)

Chạy trước khi bấm **Save Version**: xem cái gì đang chiếm chỗ trong 20GB output và
kiểm tra embedding đã đủ shard chưa.


In [30]:
#@title Kiểm kê artifact + hướng dẫn tái dùng
import os, glob

def du(path):
    tot = 0
    for r, _, fs in os.walk(path):
        for f in fs:
            try: tot += os.path.getsize(os.path.join(r, f))
            except OSError: pass
    return tot

print('ART =', ART)
rows = []
for p in sorted(glob.glob(f'{ART}/*')) + sorted(glob.glob(f'{ART}/emb/*')):
    if p.rstrip('/').endswith('/emb'):
        continue
    n = len(glob.glob(f'{p}/*')) if os.path.isdir(p) else 1
    rows.append((du(p) if os.path.isdir(p) else os.path.getsize(p), n, p))
for sz, n, p in sorted(rows, reverse=True):
    print(f'  {sz/1e6:9.1f} MB  {n:5d} file  {p.replace(ART + "/", "")}')
print(f'  {du(ART)/1e6:9.1f} MB  TỔNG' + (' / 20000 MB giới hạn Kaggle' if ON_KAGGLE else ''))

if ON_KAGGLE:
    print("""
Quy trình 2 Version (RUN_PHASE hiện tại = {RUN_PHASE}):
  • Version 1 phải chạy với RUN_PHASE=1 và output phải có models/reranker-ft-v3b_ft_reranker_improved.
  • Version 2 đổi RUN_PHASE=2 và Add Input dataset `v3b-finetune-reranker-improve`.

Để session sau đọc artifact:
  1. Add Input dataset `kitnehi1211/v3b-finetune-reranker-improve`
  2. Stage 0 đọc trực tiếp thư mục artifact cố định trong dataset
  3. Stage 5 chỉ encode shard còn thiếu

Nếu output vượt 20GB, xoá bớt embedding của profile không dùng nữa bằng lệnh:
  (thêm dấu ! ở đầu)  rm -rf {ART}/emb/<tag-cũ>""".replace('{ART}', ART).replace('{RUN_PHASE}', str(RUN_PHASE)))
else:
    print('\nColab: artifact đã nằm trên Drive, không cần làm gì thêm.')


ART = /kaggle/working/DSC2026_Task1/artifacts
        0.0 MB      1 file  experiments.csv
        0.0 MB      0 file  models
        0.0 MB      0 file  emb/biencoder_ft_v3b_ft_reranker_improved_c3000_m2500_t1
        0.0 MB  TỔNG / 20000 MB giới hạn Kaggle

Quy trình 2 Version (RUN_PHASE hiện tại = 2):
  • Version 1 phải chạy với RUN_PHASE=1 và output phải có models/reranker-ft-v3b_ft_reranker_improved.
  • Version 2 đổi RUN_PHASE=2 và Add Input dataset `v3b-finetune-reranker-improve`.

Để session sau đọc artifact:
  1. Add Input dataset `kitnehi1211/v3b-finetune-reranker-improve`
  2. Stage 0 đọc trực tiếp thư mục artifact cố định trong dataset
  3. Stage 5 chỉ encode shard còn thiếu

Nếu output vượt 20GB, xoá bớt embedding của profile không dùng nữa bằng lệnh:
  (thêm dấu ! ở đầu)  rm -rf /kaggle/working/DSC2026_Task1/artifacts/emb/<tag-cũ>


---
## Việc cần làm ngoài notebook

**1. Đăng ký model với BTC — bắt buộc, hạn 18/09.**
Bài nộp dùng model chưa được phê duyệt sẽ **không được công nhận**.
[Form đăng ký](https://forms.gle/HWE7tcxzWq63Kxv28) ·
[Danh sách đã duyệt](https://docs.google.com/spreadsheets/d/1c5jzsYezWho1WGLRfMKWOaFPLIk_GTnXP5vV8AOWM2Q/edit)

Notebook này dùng: `AITeamVN/Vietnamese_Embedding_v2` (0.6B) +
`AITeamVN/Vietnamese_Reranker` (0.6B) = **1.2B < 4B**. Cả hai đã có trong danh sách BTC duyệt. Nên đăng ký rộng hơn thực dùng ngay
từ đầu (thêm vài phương án dự phòng) vì chờ duyệt mất thời gian — và nếu bạn chạy cả hai
notebook thì phải đăng ký **cả 4 model**.

**2. Nộp bằng đúng Organization `SGU_LegalMind`** trên Codabench. Bài nộp không qua
Organization là **không hợp lệ**.

**3. Ngân sách 10 lượt nộp/ngày.** Tune hoàn toàn trên val offline, chỉ nộp khi val có cải
thiện thật. 1.000 câu public là tập nhỏ — chênh 1–2% rất có thể chỉ là nhiễu, đừng
overfit leaderboard, vì private test mới là điểm ăn thua.

## Thứ tự chạy đề xuất

| # | Việc | GPU? | Kỳ vọng |
|---|---|---|---|
| 1 | Stage 1–4 (BM25) | **không** | so với TF-IDF `recall@5=0.486` |
| 3 | Stage 5 encode + dense | có | `recall@100` nên > 0.90 |
| 4 | Stage 6 RRF | có | tăng nhẹ so với từng nhánh |
| 5 | Stage 7 rerank → **nộp lần đầu** | có | bước tăng mạnh nhất |
| 6 | Stage 8 fine-tune → encode lại → nộp | có | bước tăng lớn thứ hai |

Chạy bước 2 trong session **CPU** để giữ quota GPU — đó là tài nguyên khan hiếm nhất
trong 38 ngày còn lại.

## Nếu muốn đẩy xa hơn
- **Fine-tune cả reranker** trên train split (thường hơn fine-tune bi-encoder).
- **Ensemble** nhiều bi-encoder (e5, gte-multilingual) rồi RRF — vẫn dư ngân sách 4B.
- **Query expansion**: câu hỏi chỉ ~20 từ, rất ngắn.
- **Doc-level prior**: 3.105/8.532 văn bản từng là gold trong train — phân bố chủ đề của
  corpus có thể khai thác được, nhưng cẩn thận overfit.
